In [ ]:
#!/usr/bin/env python3
"""
BiLSTM-CNN-CRF training script for HK Address Parser — Version 7 (Grid Search)
=============================================================================
Key changes vs V6:
- Full grid search over a pruned, high-value hyperparameter space
- PRIMARY early-stopping & model selection metric = Test Logic Acc
  (evaluated on address_dataset.jsonl which only has line1 / line2)
- Validation entity F1 / val loss are logged for monitoring only
- Every epoch of every trial is saved
- Each trial has its own directory
- Final ranking of all trials by best Test Logic Acc
"""

import os
import re
import sys
import json
import math
import time
import itertools
import torch
import subprocess
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
from torchcrf import CRF
from collections import defaultdict, Counter

torch.backends.cudnn.enabled = False

# ====================== Fixed paths ======================
TRAIN_FILE = "data2/train_cleaned.jsonl"
VAL_FILE   = "data2/validation_cleaned.jsonl"
TEST_FILE  = "data2/address_dataset.jsonl"
BASE_OUTPUT_DIR = "./bilstm_crf_modelV7_search"

# ====================== Grid Search Space (pruned) ======================
# Keep this small. You can expand later after seeing which region works.
GRID = {
    "learning_rate":   [1.5e-4, 2.5e-4, 3.5e-4],
    "dropout":         [0.40, 0.50, 0.60],
    "weight_decay":    [1e-5, 3e-5, 1e-4],
    "hidden_dim":      [256, 320],
    "num_layers":      [1, 2],
    "char_cnn_filters":[100],          # keep even feature_dim; expand later if needed
    "batch_size":      [256],
}

# Training defaults that are the same for every trial
EPOCHS               = 30
EARLY_STOP_PATIENCE  = 3          # based on Test Logic Acc
MIN_EPOCHS_BEFORE_STOP = 8        # never stop before this
WORD_EMBED_DIM       = 300
CHAR_EMBED_DIM       = 100
ROPE_BASE            = 10000.0
MAX_SEQ_LEN_ROPE     = 512
TEST_EVAL_EVERY      = 1          # evaluate test every N epochs (1 = every epoch)

# ====================== GPU helper ======================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
             "--format=csv,nounits,noheader"], encoding="utf-8")
        best_id, max_free = -1, 0
        fallback_id, fallback_free = 0, 0
        for line in result.strip().split("\n"):
            parts = line.split(", ")
            gid, free, util = int(parts[0]), int(parts[1]), int(parts[2])
            if free > fallback_free:
                fallback_free, fallback_id = free, gid
            if util < 30 and free > max_free:
                max_free, best_id = free, gid
        if best_id != -1:
            print(f"--> Selected GPU {best_id} ({max_free} MB free)")
            return torch.device(f"cuda:{best_id}")
        print(f"⚠️ Falling back to GPU {fallback_id}")
        return torch.device(f"cuda:{fallback_id}")
    except Exception as e:
        print(f"⚠️ nvidia-smi failed: {e}. Using cuda:0")
        return torch.device("cuda:0")

# ====================== Data helpers (unchanged logic) ======================
def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)
    field_to_tag = {
        "flat": "UNIT", "floor": "FLOOR", "block": "BLOCK", "phase": "PHASE",
        "building_name": "BUILDING_NAME", "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME", "sub_district": "SUB_DISTRICT",
        "district": "DISTRICT", "region": "REGION",
        "village_name": "VILLAGE_NAME", "building_number": "BUILDING_NUMBER"
    }
    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict
    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)
    for field, tag, term in items_to_process:
        start_idx = 0
        while True:
            idx = input_text.find(term, start_idx)
            if idx == -1:
                break
            is_already_tagged = any(char_labels[i] != "O" for i in range(idx, idx + len(term)))
            if not is_already_tagged:
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break
            else:
                start_idx = idx + 1
    return char_labels

def tokenize_and_align(input_text, output_dict):
    char_labels = reconstruct_char_labels(input_text, output_dict)
    tokens, token_tags = [], []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        token_str = match.group()
        start_idx = match.start()
        tokens.append(token_str)
        token_tags.append(char_labels[start_idx])
    return tokens, token_tags

class Vocab:
    def __init__(self):
        self.w2i = {"<PAD>": 0, "<UNK>": 1}
        self.i2w = {0: "<PAD>", 1: "<UNK>"}
    def add(self, word):
        if word not in self.w2i:
            idx = len(self.w2i)
            self.w2i[word] = idx
            self.i2w[idx] = word
    def __len__(self):
        return len(self.w2i)

class AddressDataset(Dataset):
    def __init__(self, data_list, word_vocab, char_vocab, tag2idx):
        self.data = data_list
        self.w2i = word_vocab
        self.c2i = char_vocab
        self.t2i = tag2idx
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        word_ids = [self.w2i.w2i.get(t, self.w2i.w2i["<UNK>"]) for t in tokens]
        char_ids_list = [[self.c2i.w2i.get(c, self.c2i.w2i["<UNK>"]) for c in token] for token in tokens]
        tag_ids = [self.t2i[tag] for tag in tags]
        return word_ids, char_ids_list, tag_ids

def collate_fn(batch):
    max_seq_len = max(len(item[0]) for item in batch)
    max_word_len = max([1] + [len(c) for item in batch for c in item[1]])
    b_words, b_chars, b_labels, b_masks = [], [], [], []
    for word_ids, char_ids_list, label_ids in batch:
        seq_len = len(word_ids)
        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_labels.append(label_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))
        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)
    return {
        "word_ids": torch.tensor(b_words, dtype=torch.long),
        "char_ids": torch.tensor(b_chars, dtype=torch.long),
        "labels": torch.tensor(b_labels, dtype=torch.long),
        "mask": torch.tensor(b_masks, dtype=torch.bool)
    }

# ====================== RoPE + Model (unchanged) ======================
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(x, cos, sin):
    return (x * cos) + (rotate_half(x) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=512, base=10000.0):
        super().__init__()
        assert dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)
    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)
        self.max_seq_len = seq_len
    def forward(self, x):
        seq_len = x.shape[1]
        if seq_len > self.max_seq_len:
            self._build_cache(seq_len)
        cos = self.cos_cached[:, :seq_len, :].to(dtype=x.dtype)
        sin = self.sin_cached[:, :seq_len, :].to(dtype=x.dtype)
        return apply_rotary_pos_emb(x, cos, sin)

class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters,
                 hidden_dim, num_layers=1, dropout=0.5, max_seq_len=512, rope_base=10000.0):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters,
                                  kernel_size=3, padding=1)
        feature_dim = word_dim + cnn_filters
        assert feature_dim % 2 == 0, f"feature_dim={feature_dim} must be even for RoPE"
        self.rotary = RotaryEmbedding(dim=feature_dim, max_seq_len=max_seq_len, base=rope_base)
        self.lstm = nn.LSTM(feature_dim, hidden_dim // 2, num_layers=num_layers,
                            bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]
        w_emb = self.word_embed(word_ids)
        char_ids_flat = char_ids.view(-1, max_word_len)
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)
        c_features = c_cnn_out.view(batch_size, seq_len, -1)
        features = torch.cat([w_emb, c_features], dim=2)
        features = self.rotary(features)
        lstm_in = self.dropout(features)
        lstm_out, _ = self.lstm(lstm_in)
        emissions = self.hidden2tag(lstm_out)
        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask), emissions

# ====================== Evaluation helpers (unchanged) ======================
def normalize_for_eval(text):
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

def extract_entities_from_bio(tag_list):
    entities = []
    i = 0
    while i < len(tag_list):
        tag = tag_list[i]
        if tag.startswith("B-"):
            ent_type = tag[2:]
            start = i
            i += 1
            while i < len(tag_list) and tag_list[i] == f"I-{ent_type}":
                i += 1
            entities.append((start, i - 1, ent_type))
        else:
            i += 1
    return entities

def compute_entity_f1(gold_tag_seqs, pred_tag_seqs):
    tp = fp = fn = 0
    per_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    for gold_tags, pred_tags in zip(gold_tag_seqs, pred_tag_seqs):
        gold_ents = set(extract_entities_from_bio(gold_tags))
        pred_ents = set(extract_entities_from_bio(pred_tags))
        tp += len(gold_ents & pred_ents)
        fp += len(pred_ents - gold_ents)
        fn += len(gold_ents - pred_ents)
        for ent in gold_ents & pred_ents:
            per_type[ent[2]]["tp"] += 1
        for ent in pred_ents - gold_ents:
            per_type[ent[2]]["fp"] += 1
        for ent in gold_ents - pred_ents:
            per_type[ent[2]]["fn"] += 1
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    per_type_f1 = {}
    for t, counts in per_type.items():
        p = counts["tp"] / (counts["tp"] + counts["fp"] + 1e-9)
        r = counts["tp"] / (counts["tp"] + counts["fn"] + 1e-9)
        per_type_f1[t] = 2 * p * r / (p + r + 1e-9)
    return {"precision": precision, "recall": recall, "f1": f1, "per_type_f1": per_type_f1}

def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
    # exact copy of your original function – do not change
    line1_keys = set()
    logic_keys_used = set()
    if "flat" in extracted_labels:
        line1_keys.add("flat")
        logic_keys_used.add("flat")
    if "floor" in extracted_labels:
        line1_keys.add("floor")
        logic_keys_used.add("floor")
    has_bldg = "building_name" in extracted_labels
    has_block = "block" in extracted_labels
    has_est = "estate_name" in extracted_labels
    has_phase = "phase" in extracted_labels
    has_vill = "village_name" in extracted_labels
    has_street = "street_name" in extracted_labels
    has_bldg_no = "building_number" in extracted_labels
    if has_block:
        line1_keys.add("block")
        logic_keys_used.add("block")
        if has_bldg and has_est:
            line1_keys.add("building_name")
            logic_keys_used.add("building_name")
        elif has_bldg and not has_est:
            pass
    elif has_bldg:
        line1_keys.add("building_name")
        logic_keys_used.add("building_name")
    elif has_est:
        line1_keys.add("estate_name")
        logic_keys_used.add("estate_name")
        if has_phase:
            line1_keys.add("phase")
            logic_keys_used.add("phase")
    elif has_vill:
        line1_keys.add("village_name")
        logic_keys_used.add("village_name")
        if has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")
    elif has_street:
        line1_keys.add("street_name")
        logic_keys_used.add("street_name")
        if has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")
    elif has_bldg_no:
        line1_keys.add("building_number")
        logic_keys_used.add("building_number")
    is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
    mapping = {
        "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
        "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
        "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
        "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
        "DISTRICT": "district", "REGION": "region"
    }
    token_groups = []
    for entity in parsed_entities:
        raw_tag = entity["entity_group"]
        mapped_tag = mapping.get(raw_tag, raw_tag.lower())
        if mapped_tag in line1_keys:
            token_groups.append("micro")
        elif mapped_tag not in ("o", "O"):
            token_groups.append("macro")
        else:
            token_groups.append("O")
    resolved_groups = []
    last_valid = "micro" if not is_chinese else "macro"
    for tg in token_groups:
        if tg != "O":
            last_valid = tg
            resolved_groups.append(tg)
        else:
            resolved_groups.append(last_valid)
    micro_segments, macro_segments = [], []
    curr_micro_seg, curr_micro_tag = [], None
    curr_macro_seg, curr_macro_tag = [], None
    for entity, group in zip(parsed_entities, resolved_groups):
        tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
        word = entity["word"]
        if group == "micro":
            if tag not in ("o", "O"):
                if curr_micro_tag != tag:
                    if curr_micro_seg:
                        micro_segments.append((curr_micro_tag, curr_micro_seg))
                    curr_micro_seg = [word]
                    curr_micro_tag = tag
                else:
                    curr_micro_seg.append(word)
            else:
                if curr_micro_seg:
                    curr_micro_seg.append(word)
                else:
                    curr_micro_seg, curr_micro_tag = [word], "o"
        else:
            if tag not in ("o", "O"):
                if curr_macro_tag != tag:
                    if curr_macro_seg:
                        macro_segments.append((curr_macro_tag, curr_macro_seg))
                    curr_macro_seg = [word]
                    curr_macro_tag = tag
                else:
                    curr_macro_seg.append(word)
            else:
                if curr_macro_seg:
                    curr_macro_seg.append(word)
                else:
                    curr_macro_seg, curr_macro_tag = [word], "o"
    if curr_micro_seg:
        micro_segments.append((curr_micro_tag, curr_micro_seg))
    if curr_macro_seg:
        macro_segments.append((curr_macro_tag, curr_macro_seg))
    micro_string = "".join("".join(words) for _, words in micro_segments)
    macro_string = "".join("".join(words) for _, words in macro_segments)
    def clean_string(s):
        s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
        s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
        return re.sub(r"\s{2,}", " ", s).strip()
    micro_string = clean_string(micro_string)
    macro_string = clean_string(macro_string)
    split_conf = 1.0
    for k in logic_keys_used:
        split_conf *= conf_mapped.get(k, 1.0)
    if not logic_keys_used:
        for k, c in conf_mapped.items():
            if k not in ("district", "region", "sub_district"):
                split_conf *= c
    line1 = macro_string if is_chinese else micro_string
    line2 = micro_string if is_chinese else macro_string
    return line1, line2, split_conf, list(logic_keys_used)

def evaluate_on_val(model, val_loader, idx2tag, device):
    model.eval()
    all_gold, all_pred = [], []
    token_correct = token_total = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=" → Entity metrics", leave=False):
            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)
            pred_ids_list, _ = model(word_ids, char_ids, mask)
            for b in range(word_ids.size(0)):
                seq_len = mask[b].sum().item()
                gold_ids = labels[b][:seq_len].tolist()
                pred_ids = pred_ids_list[b][:seq_len]
                gold_tags = [idx2tag[i] for i in gold_ids]
                pred_tags = [idx2tag[i] for i in pred_ids]
                all_gold.append(gold_tags)
                all_pred.append(pred_tags)
                for g, p in zip(gold_ids, pred_ids):
                    token_total += 1
                    if g == p:
                        token_correct += 1
    token_acc = token_correct / (token_total + 1e-9)
    entity_metrics = compute_entity_f1(all_gold, all_pred)
    return token_acc, entity_metrics

def evaluate_on_test(model, test_data, word_vocab, char_vocab, tag2idx, idx2tag, device,
                     batch_size=32, max_samples=None):
    """This is the REAL target metric. Only uses line1 / line2 strings."""
    model.eval()
    if max_samples:
        test_data = test_data[:max_samples]
    mapping = {
        "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
        "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
        "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
        "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
        "DISTRICT": "district", "REGION": "region"
    }
    stats = {"logic": 0, "line1": 0, "line2": 0, "full": 0, "total": 0, "excluded": 0}
    ALL_FIELDS = [
        "flat", "floor", "building_name", "block", "phase",
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]
    def content_covered(gt_text, addr_norm):
        gt_norm = normalize_for_eval(gt_text)
        if not gt_norm:
            return True
        gt_cnt = Counter(gt_norm)
        addr_cnt = Counter(addr_norm)
        return all(addr_cnt[c] >= cnt for c, cnt in gt_cnt.items())

    valid_items = []
    for item in test_data:
        address = item["input"].strip()
        out_raw = item.get("output", {})
        l1_raw = out_raw.get("line1", "")
        l2_raw = out_raw.get("line2", "")
        has_tags = isinstance(l1_raw, dict) or isinstance(l2_raw, dict)
        if has_tags:
            gt_tags = {}
            if isinstance(l1_raw, dict):
                gt_tags.update({k: v for k, v in l1_raw.items() if v})
            if isinstance(l2_raw, dict):
                gt_tags.update({k: v for k, v in l2_raw.items() if v})
            gt_l1, gt_l2 = "", ""
        else:
            gt_tags = {}
            gt_l1 = str(l1_raw).strip()
            gt_l2 = str(l2_raw).strip()
        norm_address = normalize_for_eval(address)
        is_corrupted = False
        if has_tags and gt_tags:
            for _, gt_val in gt_tags.items():
                if normalize_for_eval(gt_val) and normalize_for_eval(gt_val) not in norm_address:
                    is_corrupted = True
                    break
        else:
            if not content_covered(gt_l1, norm_address) or not content_covered(gt_l2, norm_address):
                is_corrupted = True
        if is_corrupted:
            stats["excluded"] += 1
            continue
        tokens = []
        for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', address):
            tokens.append(match.group())
        if not tokens:
            continue
        valid_items.append({
            "address": address, "tokens": tokens, "has_tags": has_tags,
            "gt_tags": gt_tags, "gt_l1": gt_l1, "gt_l2": gt_l2
        })

    for i in tqdm(range(0, len(valid_items), batch_size), desc=" → Test set", leave=False):
        batch_items = valid_items[i:i + batch_size]
        batch_tokens = [x["tokens"] for x in batch_items]
        max_seq_len = max(len(t) for t in batch_tokens) if batch_tokens else 1
        max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])
        b_words, b_chars, b_masks = [], [], []
        for tokens in batch_tokens:
            seq_len = len(tokens)
            word_ids = [word_vocab.w2i.get(t, 1) for t in tokens]
            char_ids_list = [[char_vocab.w2i.get(c, 1) for c in token] for token in tokens]
            b_words.append(word_ids + [0] * (max_seq_len - seq_len))
            b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))
            padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
            padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
            b_chars.append(padded_chars)
        word_tensor = torch.tensor(b_words, dtype=torch.long).to(device)
        char_tensor = torch.tensor(b_chars, dtype=torch.long).to(device)
        mask_tensor = torch.tensor(b_masks, dtype=torch.bool).to(device)
        with torch.no_grad():
            pred_ids_batch, emissions_batch = model(word_tensor, char_tensor, mask_tensor)
            batch_probs = torch.softmax(emissions_batch, dim=-1).cpu()
        for idx, item_data in enumerate(batch_items):
            address = item_data["address"]
            tokens = item_data["tokens"]
            pred_ids = pred_ids_batch[idx]
            probs = batch_probs[idx]
            char_tags = ["O"] * len(address)
            char_confs = [0.0] * len(address)
            start_indices = [m.start() for m in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', address)]
            for j, tag_id in enumerate(pred_ids):
                if j >= len(start_indices):
                    break
                start_idx = start_indices[j]
                tag = idx2tag[tag_id]
                conf = probs[j, tag_id].item()
                token_len = len(tokens[j])
                for c in range(start_idx, min(start_idx + token_len, len(char_tags))):
                    if char_tags[c] == "O":
                        char_tags[c] = tag
                        char_confs[c] = conf
            parsed_entities = []
            for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address):
                token_str = match.group(1)
                trailing = match.group(2)
                start_idx = match.start(1)
                tag = char_tags[start_idx] if start_idx < len(char_tags) else "O"
                conf = char_confs[start_idx] if start_idx < len(char_confs) else 0.0
                entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                parsed_entities.append({"entity_group": entity_group, "word": token_str + trailing, "conf": conf})
            components = defaultdict(list)
            confs = defaultdict(list)
            for ent in parsed_entities:
                if ent["entity_group"] == "O":
                    continue
                components[ent["entity_group"]].append(ent["word"])
                confs[ent["entity_group"]].append(ent["conf"])
            extracted_raw = {k: "".join(v) for k, v in components.items()}
            conf_raw = {k: sum(v) / len(v) for k, v in confs.items()}
            extracted_mapped = {mapping.get(k, k.lower()): v for k, v in extracted_raw.items()}
            conf_mapped = {mapping.get(k, k.lower()): v for k, v in conf_raw.items()}
            line1, line2, split_conf, used_keys = _split_address(
                extracted_mapped, address, parsed_entities, conf_mapped
            )
            has_tags = item_data["has_tags"]
            gt_tags = item_data["gt_tags"]
            if has_tags and gt_tags:
                pred_ff = normalize_for_eval(extracted_mapped.get("floor", "") + extracted_mapped.get("flat", ""))
                gt_ff = normalize_for_eval(gt_tags.get("floor", "") + gt_tags.get("flat", ""))
                p_bldg = normalize_for_eval(extracted_mapped.get("building_name", ""))
                p_est = normalize_for_eval(extracted_mapped.get("estate_name", ""))
                g_bldg = normalize_for_eval(gt_tags.get("building_name", ""))
                g_est = normalize_for_eval(gt_tags.get("estate_name", ""))
                field_correct = {}
                for field in ALL_FIELDS:
                    p_norm = normalize_for_eval(extracted_mapped.get(field, ""))
                    g_norm = normalize_for_eval(gt_tags.get(field, ""))
                    if p_norm == g_norm:
                        field_correct[field] = True
                    elif field in ("floor", "flat") and pred_ff == gt_ff and pred_ff != "":
                        field_correct[field] = True
                    elif field in ("building_name", "estate_name") and (p_bldg == g_est and p_est == g_bldg):
                        field_correct[field] = True
                    else:
                        field_correct[field] = False
                is_logic_correct = all(field_correct.get(k, False) for k in used_keys) if used_keys else True
                is_l1_correct = all(field_correct[f] for f in ["flat", "floor", "block", "phase", "building_name", "estate_name"])
                is_l2_correct = all(field_correct[f] for f in ["village_name", "building_number", "street_name", "sub_district", "district", "region"])
                is_full_correct = is_l1_correct and is_l2_correct
            else:
                l1_ok = (normalize_for_eval(line1) == normalize_for_eval(item_data["gt_l1"]))
                l2_ok = (normalize_for_eval(line2) == normalize_for_eval(item_data["gt_l2"]))
                is_l1_correct = l1_ok
                is_l2_correct = l2_ok
                is_full_correct = l1_ok and l2_ok
                is_logic_correct = is_full_correct
            stats["total"] += 1
            if is_logic_correct:
                stats["logic"] += 1
            if is_l1_correct:
                stats["line1"] += 1
            if is_l2_correct:
                stats["line2"] += 1
            if is_full_correct:
                stats["full"] += 1
    if stats["total"] == 0:
        return None
    return {
        "logic_acc": stats["logic"] / stats["total"],
        "line1_acc": stats["line1"] / stats["total"],
        "line2_acc": stats["line2"] / stats["total"],
        "full_acc": stats["full"] / stats["total"],
        "total": stats["total"],
        "excluded": stats["excluded"]
    }

# ====================== Metrics writer ======================
def _write_epoch_metrics(log_path, jsonl_path, epoch, config,
                         train_loss, val_loss, token_acc, ent_metrics, test_metrics,
                         was_best_logic, best_test_logic, patience):
    lines = []
    lines.append(f"{'─'*90}")
    lines.append(f"Epoch {epoch:02d} | config = {config}")
    lines.append(f"{'─'*90}")
    lines.append(f" Train loss     : {train_loss:.6f}")
    if val_loss is not None:
        lines.append(f" Val loss       : {val_loss:.6f}")
    if token_acc is not None:
        lines.append(f" Token accuracy : {token_acc*100:.2f}%")
    if ent_metrics is not None:
        lines.append(f" Entity F1      : {ent_metrics['f1']*100:.2f}%")
    if test_metrics is not None:
        lines.append(f" ★ Test Logic Acc : {test_metrics['logic_acc']*100:.2f}%")
        lines.append(f"   Test Line1 Acc : {test_metrics['line1_acc']*100:.2f}%")
        lines.append(f"   Test Line2 Acc : {test_metrics['line2_acc']*100:.2f}%")
        lines.append(f"   Test Full Acc  : {test_metrics['full_acc']*100:.2f}%")
        lines.append(f"   (valid/excl)   : {test_metrics['total']} / {test_metrics['excluded']}")
    lines.append(f" Marked best by Logic? : {'YES ★' if was_best_logic else 'no'}")
    lines.append(f" Running best Test Logic : {best_test_logic*100:.2f}%")
    lines.append(f" Patience              : {patience}")
    lines.append("")
    with open(log_path, "a", encoding="utf-8") as f:
        f.write("\n".join(lines))

    record = {
        "epoch": epoch,
        "config": config,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "token_acc": token_acc,
        "entity_f1": ent_metrics["f1"] if ent_metrics else None,
        "test_logic_acc": test_metrics["logic_acc"] if test_metrics else None,
        "test_line1_acc": test_metrics["line1_acc"] if test_metrics else None,
        "test_line2_acc": test_metrics["line2_acc"] if test_metrics else None,
        "test_full_acc": test_metrics["full_acc"] if test_metrics else None,
        "was_best_logic": was_best_logic,
        "best_test_logic": best_test_logic,
        "patience": patience,
    }
    with open(jsonl_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# ====================== One full trial ======================
def run_one_trial(trial_id, config, device, word_vocab, char_vocab, tag2idx, idx2tag,
                  train_data, val_data, test_raw):
    trial_name = (f"trial_{trial_id:03d}_lr{config['learning_rate']}"
                  f"_do{config['dropout']}_wd{config['weight_decay']}"
                  f"_h{config['hidden_dim']}_l{config['num_layers']}")
    trial_dir = os.path.join(BASE_OUTPUT_DIR, trial_name)
    os.makedirs(trial_dir, exist_ok=True)

    # ---------- 1. Fully finished trial → skip completely ----------
    result_path = os.path.join(trial_dir, "trial_result.json")
    if os.path.exists(result_path):
        print(f"⏩ Skipping {trial_name} (already finished)")
        with open(result_path, "r", encoding="utf-8") as f:
            return json.load(f)

    print(f"\n{'='*100}")
    print(f"▶ Starting / Resuming {trial_name}")
    print(f" Config: {config}")
    print(f"{'='*100}\n")

    # data loaders
    train_loader = DataLoader(
        AddressDataset(train_data, word_vocab, char_vocab, tag2idx),
        batch_size=config["batch_size"], shuffle=True, collate_fn=collate_fn, num_workers=0
    )
    val_loader = DataLoader(
        AddressDataset(val_data, word_vocab, char_vocab, tag2idx),
        batch_size=config["batch_size"] * 2, shuffle=False, collate_fn=collate_fn, num_workers=0
    ) if val_data else None

    model = BiLSTM_CNN_CRF(
        vocab_size=len(word_vocab),
        char_vocab_size=len(char_vocab),
        num_tags=len(tag2idx),
        word_dim=WORD_EMBED_DIM,
        char_dim=CHAR_EMBED_DIM,
        cnn_filters=config["char_cnn_filters"],
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        max_seq_len=MAX_SEQ_LEN_ROPE,
        rope_base=ROPE_BASE
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=config["learning_rate"],
                      weight_decay=config["weight_decay"])
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    # ---------- 2. Resume from latest checkpoint if it exists ----------
    start_epoch = 0
    best_test_logic = 0.0
    patience_counter = 0
    best_epoch = 0

    # Find the highest checkpoint_epoch_XX.pt
    existing_ckpts = []
    for f in os.listdir(trial_dir):
        if f.startswith("checkpoint_epoch_") and f.endswith(".pt"):
            try:
                ep = int(f[len("checkpoint_epoch_"):-3])
                existing_ckpts.append((ep, f))
            except ValueError:
                pass
    
    if existing_ckpts:
        existing_ckpts.sort()
        last_ep, last_ckpt_name = existing_ckpts[-1]
        ckpt_path = os.path.join(trial_dir, last_ckpt_name)
        print(f"🔄 Found checkpoint → {last_ckpt_name}. Resuming from epoch {last_ep + 1}")
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1

        # 🩹 ULTIMATE FIX: Recalculate true stats from the raw scores in metrics.jsonl
        metrics_path = os.path.join(trial_dir, "metrics.jsonl")
        if os.path.exists(metrics_path):
            try:
                with open(metrics_path, "r", encoding="utf-8") as f:
                    lines = [line.strip() for line in f if line.strip()]
                
                true_best_logic = 0.0
                true_best_epoch = 0
                true_patience = 0
                
                # Replay history to find the absolute truth
                for line in lines:
                    rec = json.loads(line)
                    ep = rec.get("epoch", 0)
                    score = rec.get("test_logic_acc")
                    
                    if score is not None:
                        if score > true_best_logic + 0.0015:
                            true_best_logic = score
                            true_best_epoch = ep
                            true_patience = 0
                        else:
                            if ep >= MIN_EPOCHS_BEFORE_STOP:
                                true_patience += 1

                # Apply the true values
                best_test_logic = true_best_logic
                best_epoch = true_best_epoch
                patience_counter = true_patience
                
                # 🩹 RESTORE WEIGHTS: Fix best_by_logic.bin if it was corrupted
                best_ckpt_path = os.path.join(trial_dir, f"checkpoint_epoch_{best_epoch:02d}.pt")
                bin_path = os.path.join(trial_dir, "best_by_logic.bin")
                
                if os.path.exists(best_ckpt_path):
                    best_ckpt = torch.load(best_ckpt_path, map_location=device)
                    torch.save(best_ckpt["model_state_dict"], bin_path)
                    print(f"✨ Restored best_by_logic.bin from TRUE best Epoch {best_epoch}!")

            except Exception as e:
                print(f"⚠️ Could not recalculate from metrics.jsonl: {e}")

        print(f" Loaded best_test_logic = {best_test_logic*100:.2f}%  "
              f"(patience={patience_counter}, best_epoch={best_epoch})")
    else:
        print("No previous checkpoint found → training from epoch 1")

    # logging (append mode)
    metrics_log = os.path.join(trial_dir, "training_metrics.log")
    metrics_jsonl = os.path.join(trial_dir, "metrics.jsonl")

    write_header = not os.path.exists(metrics_log)
    with open(metrics_log, "a", encoding="utf-8") as f:
        if write_header:
            f.write(f"Trial {trial_name}\nConfig: {json.dumps(config)}\n\n")
        else:
            f.write(f"\n\n===== RESUMED at {time.strftime('%Y-%m-%d %H:%M:%S')} "
                    f"(from epoch {start_epoch+1}) =====\n\n")
    open(metrics_jsonl, "a").close()
    # -------------------------------------------------------

    for epoch in range(start_epoch, EPOCHS):
        # ---------- TRAIN ----------
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"[{trial_name}] Ep {epoch+1}/{EPOCHS} Train", leave=False)
        for batch in pbar:
            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels   = batch["labels"].to(device)
            mask     = batch["mask"].to(device)
            optimizer.zero_grad()
            loss = model(word_ids, char_ids, mask, labels=labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        avg_train_loss = total_loss / len(train_loader)

        # ❌ REMOVED CHECKPOINT SAVING FROM HERE ❌

        # ---------- VAL (monitoring only) ----------
        avg_val_loss = None
        token_acc = None
        ent_metrics = None
        if val_loader:
            model.eval()
            total_val = 0.0
            with torch.no_grad():
                val_pbar = tqdm(val_loader, desc=f"[{trial_name}] Ep {epoch+1} Val Loss", leave=False)
                for batch in val_pbar:
                    word_ids = batch["word_ids"].to(device)
                    char_ids = batch["char_ids"].to(device)
                    labels = batch["labels"].to(device)
                    mask = batch["mask"].to(device)
                    loss = model(word_ids, char_ids, mask, labels=labels).item()
                    total_val += loss
                    val_pbar.set_postfix(loss=f"{loss:.4f}")
            avg_val_loss = total_val / len(val_loader)
            token_acc, ent_metrics = evaluate_on_val(model, val_loader, idx2tag, device)
            scheduler.step(avg_val_loss)

        # ---------- TEST (PRIMARY METRIC) ----------
        test_metrics = None
        was_best_logic = False
        if (epoch + 1) % TEST_EVAL_EVERY == 0 or epoch + 1 >= MIN_EPOCHS_BEFORE_STOP:
            test_metrics = evaluate_on_test(
                model, test_raw, word_vocab, char_vocab, tag2idx, idx2tag, device
            )
            if test_metrics:
                current_logic = test_metrics["logic_acc"]
                print(f"  → Test Logic Acc = {current_logic*100:.2f}%")

                if current_logic > best_test_logic + 0.0015:   # meaningful improvement
                    best_test_logic = current_logic
                    best_epoch = epoch + 1
                    patience_counter = 0
                    was_best_logic = True
                    # save the best-by-logic weights
                    torch.save(model.state_dict(),
                               os.path.join(trial_dir, "best_by_logic.bin"))
                    with open(os.path.join(trial_dir, "vocabs.json"), "w", encoding="utf-8") as f:
                        json.dump({"w2i": word_vocab.w2i, "c2i": char_vocab.w2i, "t2i": tag2idx}, f)
                    print(f"  ★ New best Test Logic Acc → saved best_by_logic.bin")
                else:
                    if epoch + 1 >= MIN_EPOCHS_BEFORE_STOP:
                        patience_counter += 1

        # log
        _write_epoch_metrics(
            metrics_log, metrics_jsonl, epoch + 1, config,
            avg_train_loss, avg_val_loss, token_acc, ent_metrics, test_metrics,
            was_best_logic, best_test_logic, patience_counter
        )

        # ✅ ADD CHECKPOINT SAVING HERE (After updates are made) ✅
        ckpt_path = os.path.join(trial_dir, f"checkpoint_epoch_{epoch+1:02d}.pt")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "best_test_logic": best_test_logic,
            "patience_counter": patience_counter,
            "best_epoch": best_epoch,
        }, ckpt_path)

        # early stop purely on Test Logic Acc
        if patience_counter >= EARLY_STOP_PATIENCE and epoch + 1 >= MIN_EPOCHS_BEFORE_STOP:
            print(f"\n🛑 Early stopping trial {trial_name} "
                  f"(no Test Logic improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

    # final summary for this trial
    result = {
        "trial_id": trial_id,
        "trial_name": trial_name,
        "config": config,
        "best_test_logic": best_test_logic,
        "best_epoch": best_epoch,
        "trial_dir": trial_dir,
    }
    with open(os.path.join(trial_dir, "trial_result.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Finished {trial_name} | Best Test Logic = {best_test_logic*100:.2f}% "
          f"(epoch {best_epoch})")
    return result

# ====================== Main search loop ======================
def main():
    device = get_emptiest_gpu_safely()
    print(f"Device: {device}")

    # tags
    tags = ["O"]
    tag_list = ["UNIT", "FLOOR", "BUILDING_NAME", "ESTATE_NAME", "STREET_NAME",
                "SUB_DISTRICT", "DISTRICT", "REGION", "VILLAGE_NAME", "BUILDING_NUMBER",
                "BLOCK", "PHASE"]
    for t in tag_list:
        tags.extend([f"B-{t}", f"I-{t}"])
    tag2idx = {t: i for i, t in enumerate(tags)}
    idx2tag = {i: t for t, i in tag2idx.items()}

    # vocab + data (built once, shared by all trials)
    word_vocab = Vocab()
    char_vocab = Vocab()

    def parse_file(path, update_vocab=False):
        data_list = []
        if not os.path.exists(path):
            return data_list
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                tokens, token_tags = tokenize_and_align(data["input"], data["output"])
                if update_vocab:
                    for token in tokens:
                        word_vocab.add(token)
                        for c in token:
                            char_vocab.add(c)
                data_list.append((tokens, token_tags))
        return data_list

    print("📂 Loading datasets (once)...")
    train_data = parse_file(TRAIN_FILE, update_vocab=True)
    val_data   = parse_file(VAL_FILE, update_vocab=False)
    test_raw = []
    if os.path.exists(TEST_FILE):
        with open(TEST_FILE, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip() and not line.startswith("#"):
                    test_raw.append(json.loads(line))

    print(f"✅ Vocab  → Words: {len(word_vocab)} | Chars: {len(char_vocab)}")
    print(f"✅ Data   → Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_raw)}")

    if len(train_data) == 0:
        raise FileNotFoundError(f"No training data in {TRAIN_FILE}")

    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

    # generate all combinations
    keys = list(GRID.keys())
    values = list(GRID.values())
    all_configs = [dict(zip(keys, combo)) for combo in itertools.product(*values)]
    print(f"\n🔍 Grid search: {len(all_configs)} trials")
    for i, c in enumerate(all_configs):
        print(f"  {i+1:02d}. {c}")

    # run all trials
    all_results = []
    for trial_id, config in enumerate(all_configs, 1):
        try:
            result = run_one_trial(
                trial_id, config, device,
                word_vocab, char_vocab, tag2idx, idx2tag,
                train_data, val_data, test_raw
            )
            all_results.append(result)
        except Exception as e:
            print(f"❌ Trial {trial_id} failed: {e}")
            continue

    # final ranking
    all_results.sort(key=lambda x: x["best_test_logic"], reverse=True)
    summary_path = os.path.join(BASE_OUTPUT_DIR, "search_summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print("\n" + "="*100)
    print("🏁 GRID SEARCH FINISHED – Ranking by Test Logic Acc")
    print("="*100)
    for rank, r in enumerate(all_results, 1):
        print(f"{rank:2d}. {r['best_test_logic']*100:5.2f}%  |  {r['trial_name']}  "
              f"(epoch {r['best_epoch']})")
    print(f"\nFull summary written to → {summary_path}")
    print("Each trial directory contains:")
    print("  - best_by_logic.bin          ← the weights you should use")
    print("  - checkpoint_epoch_XX.pt     ← every epoch")
    print("  - training_metrics.log / metrics.jsonl")
    print("  - vocabs.json")

if __name__ == "__main__":
    main()

--> Selected GPU 0 (11199 MB free)
Device: cuda:0
📂 Loading datasets (once)...
✅ Vocab  → Words: 29312 | Chars: 3068
✅ Data   → Train: 314210 | Val: 39276 | Test: 1803

🔍 Grid search: 108 trials
  01. {'learning_rate': 0.00015, 'dropout': 0.4, 'weight_decay': 1e-05, 'hidden_dim': 256, 'num_layers': 1, 'char_cnn_filters': 100, 'batch_size': 256}
  02. {'learning_rate': 0.00015, 'dropout': 0.4, 'weight_decay': 1e-05, 'hidden_dim': 256, 'num_layers': 2, 'char_cnn_filters': 100, 'batch_size': 256}
  03. {'learning_rate': 0.00015, 'dropout': 0.4, 'weight_decay': 1e-05, 'hidden_dim': 320, 'num_layers': 1, 'char_cnn_filters': 100, 'batch_size': 256}
  04. {'learning_rate': 0.00015, 'dropout': 0.4, 'weight_decay': 1e-05, 'hidden_dim': 320, 'num_layers': 2, 'char_cnn_filters': 100, 'batch_size': 256}
  05. {'learning_rate': 0.00015, 'dropout': 0.4, 'weight_decay': 3e-05, 'hidden_dim': 256, 'num_layers': 1, 'char_cnn_filters': 100, 'batch_size': 256}
  06. {'learning_rate': 0.00015, 'dropout': 0

[trial_002_lr0.00015_do0.4_wd1e-05_h256_l2] Ep 11/30 Train:  16%|█▌        | 197/1228 [01:36<07:18,  2.35it/s, loss=0.4472]

In [19]:
#!/usr/bin/env python3
"""
Batch Evaluation Script for BiLSTM-CNN-CRF (compatible with bilstm_crf_modelV5)
Includes Multi-Threshold Confidence Diagnostics (Baseline, 50%, 60%, 70%, 80%).
Features: Calibrated Accuracy, Situation-Aware Confidence,
Combined Floor+Flat Evaluation, Building↔Estate interchangeability,
and Space/Punctuation-Insensitive Matching.
"""
import os
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
from torchcrf import CRF

# ==========================================
# CONFIGURATION (exact match to training V5)
# ==========================================
MODEL_DIR = "./bilstm_crf_modelV5"
LOG_FILE = "parsing_results_bilstm_V5.log"
TEST_FILE = "data2/test_cleaned.jsonl"
BATCH_SIZE = 64

WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 256
NUM_LSTM_LAYERS = 1
POS_EMBED_DIM = 16
DROPOUT = 0.5

EXCLUDE_FROM_OVERALL = {"DISTRICT", "REGION", "SUB_DISTRICT"}
THRESHOLDS = [0.0, 0.50, 0.60, 0.70, 0.80]

# ==========================================
# MODEL ARCHITECTURE (exact match to training V5)
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters,
                 hidden_dim, num_layers=1, pos_dim=16, dropout=0.5, max_seq_len=512):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.pos_embed = nn.Embedding(max_seq_len, pos_dim)

        lstm_input_dim = word_dim + cnn_filters + pos_dim
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers,
                            bidirectional=True, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]

        w_emb = self.word_embed(word_ids)

        char_ids_flat = char_ids.view(-1, max_word_len)
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)
        c_features = c_cnn_out.view(batch_size, seq_len, -1)

        positions = torch.arange(seq_len, device=word_ids.device).unsqueeze(0).expand(batch_size, -1)
        pos_emb = self.pos_embed(positions)

        lstm_in = self.dropout(torch.cat([w_emb, c_features, pos_emb], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        emissions = self.hidden2tag(lstm_out)

        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')

        tags = self.crf.decode(emissions, mask=mask)
        return tags, emissions

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        return -1
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,nounits,noheader"],
            encoding="utf-8"
        )
        best_id, max_free_mb = 0, -1
        for line in result.strip().split("\n"):
            if not line.strip():
                continue
            gpu_id, free_memory = map(int, line.split(", "))
            if free_memory > max_free_mb:
                max_free_mb, best_id = free_memory, gpu_id
        return best_id
    except Exception:
        return 0

def tokenize_text(input_text):
    """Tokenize exactly as done in training."""
    tokens = []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        tokens.append(match.group())
    return tokens

def prepare_batch(batch_texts, w2i, c2i):
    batch_tokens = [tokenize_text(text) for text in batch_texts]
    max_seq_len = max(len(t) for t in batch_tokens) if batch_tokens else 1
    max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])

    b_words, b_chars, b_masks = [], [], []
    for tokens in batch_tokens:
        seq_len = len(tokens)
        word_ids = [w2i.get(t, w2i.get("<UNK>", 1)) for t in tokens]
        char_ids_list = [[c2i.get(c, c2i.get("<UNK>", 1)) for c in token] for token in tokens]

        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))

        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)

    return (
        torch.tensor(b_words, dtype=torch.long),
        torch.tensor(b_chars, dtype=torch.long),
        torch.tensor(b_masks, dtype=torch.bool),
        batch_tokens
    )

def extract_3d_components(parsed_entities):
    components = defaultdict(list)
    confs = defaultdict(list)
    for entity in parsed_entities:
        tag = entity["entity_group"]
        if tag == "O":
            continue
        word = entity["word"].strip()
        if word:
            components[tag].append(word)
            confs[tag].append(entity["conf"])

    formatted_output, conf_output = {}, {}
    for tag, words in components.items():
        joined_string = "".join(words)
        if any("\u4e00" <= char <= "\u9fff" for char in joined_string):
            formatted_output[tag] = "".join(words)
        else:
            joined_en = " ".join(words).strip()
            formatted_output[tag] = re.sub(r"\s*([/\.-])\s*", r"\1", joined_en)
        conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0
    return formatted_output, conf_output

def assemble_compact_json(extracted_data):
    return {
        "line1": {
            "flat": extracted_data.get("UNIT", ""),
            "floor": extracted_data.get("FLOOR", ""),
            "block": extracted_data.get("BLOCK", ""),
            "phase": extracted_data.get("PHASE", ""),
            "building_name": extracted_data.get("BUILDING_NAME", ""),
        },
        "line2": {
            "estate_name": extracted_data.get("ESTATE_NAME", ""),
            "village_name": extracted_data.get("VILLAGE_NAME", ""),
            "building_number": extracted_data.get("BUILDING_NUMBER", ""),
            "street_name": extracted_data.get("STREET_NAME", ""),
            "sub_district": extracted_data.get("SUB_DISTRICT", ""),
            "district": extracted_data.get("DISTRICT", ""),
            "region": extracted_data.get("REGION", ""),
        }
    }

def flatten_json(output_dict):
    flat = {}
    if "line1" in output_dict:
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def compute_situation_aware_confidence(conf_mapped):
    core = set()
    for key in ["UNIT", "FLOOR", "BLOCK", "PHASE", "ESTATE_NAME", "BUILDING_NAME",
                "VILLAGE_NAME", "STREET_NAME", "BUILDING_NUMBER"]:
        if key in conf_mapped:
            core.add(key)

    overall = 1.0
    used = []
    for k in core:
        if k in conf_mapped and k not in EXCLUDE_FROM_OVERALL and conf_mapped[k] > 0:
            overall *= conf_mapped[k]
            used.append(k)

    if not used:
        for k, c in conf_mapped.items():
            if k not in EXCLUDE_FROM_OVERALL and c > 0:
                overall *= c
                used.append(k)
    return overall

def normalize_for_eval(text):
    """Exact match to training script."""
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    device_id = get_emptiest_gpu_safely()
    device = torch.device(f"cuda:{device_id}" if device_id != -1 else "cpu")

    print(f"DEBUG: Loading Vocabs and Model from {MODEL_DIR}...")
    vocab_path = os.path.join(MODEL_DIR, "vocabs.json")
    weights_path = os.path.join(MODEL_DIR, "pytorch_model.bin")

    if not os.path.exists(vocab_path) or not os.path.exists(weights_path):
        print(f"❌ Error: Model or vocabs not found in {MODEL_DIR}.")
        return

    with open(vocab_path, "r", encoding="utf-8") as f:
        vocabs = json.load(f)
    w2i = vocabs["w2i"]
    c2i = vocabs["c2i"]
    t2i = vocabs["t2i"]
    idx2tag = {int(v): k for k, v in t2i.items()}

    model = BiLSTM_CNN_CRF(
        vocab_size=len(w2i),
        char_vocab_size=len(c2i),
        num_tags=len(t2i),
        word_dim=WORD_EMBED_DIM,
        char_dim=CHAR_EMBED_DIM,
        cnn_filters=CHAR_CNN_FILTERS,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LSTM_LAYERS,
        pos_dim=POS_EMBED_DIM,
        dropout=DROPOUT
    )
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()

    all_fields = [
        "flat", "floor", "building_name", "block", "phase",
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]

    tag_map = {
        "flat": "UNIT", "floor": "FLOOR", "block": "BLOCK",
        "phase": "PHASE", "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME", "village_name": "VILLAGE_NAME",
        "building_number": "BUILDING_NUMBER", "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT", "district": "DISTRICT", "region": "REGION",
    }

    total_time = 0.0
    stats = {t: {f: {'TP': 0, 'FP': 0, 'FN': 0} for f in all_fields} for t in THRESHOLDS}
    exact_matches = {t: 0 for t in THRESHOLDS}
    total_in_bin = {t: 0 for t in THRESHOLDS}

    print("🚀 Running batch evaluation...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        raw_lines = [line.strip() for line in file if line.strip() and not line.startswith('#')]

    test_data = [json.loads(line) for line in raw_lines]

    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Batches"):
            batch = test_data[i:i + BATCH_SIZE]
            batch_texts = [item["input"].strip() for item in batch]

            start_time = time.perf_counter()

            word_ids, char_ids, masks, batch_tokens = prepare_batch(batch_texts, w2i, c2i)
            word_ids = word_ids.to(device)
            char_ids = char_ids.to(device)
            masks = masks.to(device)

            with torch.no_grad():
                batch_predictions, batch_emissions = model(word_ids, char_ids, masks)
                batch_probs = torch.softmax(batch_emissions, dim=-1).cpu()

            total_time += time.perf_counter() - start_time

            for b_idx, item in enumerate(batch):
                address = batch_texts[b_idx]
                ground_truth_flat = flatten_json(item.get("output", {}))

                prediction_ids = batch_predictions[b_idx]
                item_probs = batch_probs[b_idx]
                tokens = batch_tokens[b_idx]

                parsed_entities = []
                for idx, tag_id in enumerate(prediction_ids):
                    token_str = tokens[idx]
                    tag = idx2tag[tag_id]
                    conf = item_probs[idx, tag_id].item()
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    parsed_entities.append({
                        "entity_group": entity_group, "word": token_str, "conf": conf,
                    })

                extracted_data, conf_output = extract_3d_components(parsed_entities)
                predicted_flat = flatten_json(assemble_compact_json(extracted_data))
                overall_conf = compute_situation_aware_confidence(conf_output)

                # --- Field evaluation (aligned with training) ---
                pred_floor_flat = normalize_for_eval(
                    predicted_flat.get("floor", "") + predicted_flat.get("flat", "")
                )
                gt_floor_flat = normalize_for_eval(
                    ground_truth_flat.get("floor", "") + ground_truth_flat.get("flat", "")
                )

                p_bldg = normalize_for_eval(predicted_flat.get("building_name", ""))
                p_est = normalize_for_eval(predicted_flat.get("estate_name", ""))
                g_bldg = normalize_for_eval(ground_truth_flat.get("building_name", ""))
                g_est = normalize_for_eval(ground_truth_flat.get("estate_name", ""))

                is_perfect_match = True
                field_evals = {}

                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")

                    pred_norm = normalize_for_eval(pred_val)
                    gt_norm = normalize_for_eval(gt_val)

                    is_correct = False
                    if pred_norm == gt_norm:
                        is_correct = True
                    elif field in ['floor', 'flat'] and pred_floor_flat == gt_floor_flat and pred_floor_flat != "":
                        is_correct = True
                    elif field in ['building_name', 'estate_name'] and (p_bldg == g_est and p_est == g_bldg):
                        is_correct = True

                    if not is_correct:
                        is_perfect_match = False

                    tp = fp = fn = 0
                    if pred_norm or gt_norm:
                        if is_correct:
                            tp = 1
                        else:
                            if pred_norm and gt_norm:
                                fp = 1
                                fn = 1
                            elif pred_norm and not gt_norm:
                                fp = 1
                            elif not pred_norm and gt_norm:
                                fn = 1

                    field_evals[field] = (tp, fp, fn)

                for t in THRESHOLDS:
                    if overall_conf >= t:
                        total_in_bin[t] += 1
                        if is_perfect_match:
                            exact_matches[t] += 1
                        for field in all_fields:
                            tp, fp, fn = field_evals[field]
                            stats[t][field]['TP'] += tp
                            stats[t][field]['FP'] += fp
                            stats[t][field]['FN'] += fn

                # --- Logging ---
                log.write(f"Original: {address}\n")
                if is_perfect_match:
                    log.write("✅ EXACT MATCH\n")
                else:
                    log.write("❌ MISMATCH FOUND\n")

                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")
                    if pred_val or gt_val:
                        tp, fp, fn = field_evals[field]
                        status = "✅" if tp == 1 else "❌"
                        tag = tag_map.get(field, field.upper())
                        conf_str = f" conf={conf_output.get(tag, 0.0):.4f}" if tag in conf_output else ""
                        log.write(f" {status} {field.upper()}:{conf_str}\n")
                        log.write(f" PRED: {pred_val if pred_val else '[None]'}\n")
                        log.write(f" TRUE: {gt_val if gt_val else '[None]'}\n")

                log.write(f"Per-label confidences : { {k: round(v, 4) for k, v in conf_output.items()} }\n")
                log.write(f"Overall confidence : {overall_conf:.6f}\n")
                log.write("-" * 50 + "\n")

    print("\n" + "=" * 65)
    print("📊 CALIBRATED METRICS EVALUATION (V5 model)")
    print("=" * 65)
    print(f"Total Addresses Tested: {len(test_data)}")
    print(f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n")

    for t in THRESHOLDS:
        title = "ALL PREDICTIONS" if t == 0.0 else f"PREDICTIONS WITH >= {int(t*100)}% CONFIDENCE"
        print("=" * 65)
        print(f"🚀 {title}")
        print("=" * 65)

        bin_count = total_in_bin[t]
        if bin_count > 0:
            exact_match_acc = (exact_matches[t] / bin_count) * 100
            print(f"Whole-Address Perfect Match : {exact_match_acc:.2f}% ({exact_matches[t]}/{bin_count})\n")

            print(f"{'FIELD':<16} | {'PRECISION':<9} | {'RECALL':<9} | {'F1-SCORE':<9} | {'SUPPORT'}")
            print("-" * 65)

            macro_f1 = 0
            valid_fields = 0

            for field in all_fields:
                tp = stats[t][field]['TP']
                fp = stats[t][field]['FP']
                fn = stats[t][field]['FN']
                support = tp + fn

                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

                if support > 0:
                    macro_f1 += f1
                    valid_fields += 1

                p_str = f"{precision*100:>5.1f}%"
                r_str = f"{recall*100:>5.1f}%"
                f1_str = f"{f1*100:>5.1f}%"
                print(f"{field:<16} | {p_str:<9} | {r_str:<9} | {f1_str:<9} | {support}")

            if valid_fields > 0:
                print("-" * 65)
                print(f"{'MACRO AVERAGE':<16} | {'-':<9} | {'-':<9} | {(macro_f1/valid_fields)*100:>5.1f}% |")
        else:
            print(f"No samples met the >= {int(t*100)}% confidence threshold.")
        print("\n")

    print(f"✅ Processing complete. Raw baseline logs saved to {LOG_FILE}")

if __name__ == "__main__":
    main()

DEBUG: Loading Vocabs and Model from ./bilstm_crf_modelV5...
🚀 Running batch evaluation...


Batches: 100%|██████████| 614/614 [02:23<00:00,  4.28it/s]



📊 CALIBRATED METRICS EVALUATION (V5 model)
Total Addresses Tested: 39277
⏱️ Total Inference runtime: 133.8966 seconds

🚀 ALL PREDICTIONS
Whole-Address Perfect Match : 98.51% (38690/39277)

FIELD            | PRECISION | RECALL    | F1-SCORE  | SUPPORT
-----------------------------------------------------------------
flat             |  99.8%    |  99.7%    |  99.7%    | 27628
floor            |  99.8%    |  99.7%    |  99.7%    | 34790
building_name    |  99.1%    |  99.0%    |  99.1%    | 24124
block            |  99.4%    |  99.3%    |  99.3%    | 5619
phase            |  98.7%    |  97.7%    |  98.2%    | 1049
estate_name      |  99.0%    |  99.0%    |  99.0%    | 11886
village_name     |  98.6%    |  98.4%    |  98.5%    | 10662
building_number  |  99.7%    |  99.7%    |  99.7%    | 38343
street_name      |  99.9%    |  99.9%    |  99.9%    | 28080
sub_district     |  99.4%    |  99.3%    |  99.4%    | 7744
district         |  99.9%    |  99.9%    |  99.9%    | 38862
region       

In [20]:
#!/usr/bin/env python3
"""
HK Address Parser - Final App with Integrated Evaluation (BiLSTM-CNN-CRF V5)
Fully compatible with the new training script (bilstm_crf_modelV5).
Uses the exact new address-split logic, new hyper-parameters and positional embeddings.
"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
from torchcrf import CRF

# ==========================================
# CONFIGURATION (exact match to training V5)
# ==========================================
MODEL_DIR = "./bilstm_crf_modelV5"
LOG_FILE = "address_split_results_bilstm_V5.log"
TEST_FILE = "data2/test_cleaned.jsonl"
BATCH_SIZE = 32

WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 256
NUM_LSTM_LAYERS = 1
POS_EMBED_DIM = 16
DROPOUT = 0.5

THRESHOLDS = [0.0, 0.2, 0.3, 0.4, 0.50, 0.60, 0.70, 0.80]

ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def flatten_json(output_dict):
    flat = {}
    if "line1" in output_dict and isinstance(output_dict["line1"], dict):
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def normalize_for_eval(text):
    """Exact match to training script."""
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

def tokenize_text(input_text):
    tokens = []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        tokens.append(match.group())
    return tokens

def prepare_batch(batch_texts, w2i, c2i):
    batch_tokens = [tokenize_text(text) for text in batch_texts]
    max_seq_len = max(len(t) for t in batch_tokens) if batch_tokens else 1
    max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])

    b_words, b_chars, b_masks = [], [], []
    for tokens in batch_tokens:
        seq_len = len(tokens)
        word_ids = [w2i.get(t, w2i.get("<UNK>", 1)) for t in tokens]
        char_ids_list = [[c2i.get(c, c2i.get("<UNK>", 1)) for c in token] for token in tokens]

        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))

        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)

    return (
        torch.tensor(b_words, dtype=torch.long),
        torch.tensor(b_chars, dtype=torch.long),
        torch.tensor(b_masks, dtype=torch.bool),
        batch_tokens
    )

# ==========================================
# MODEL (exact match to training V5)
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters,
                 hidden_dim, num_layers=1, pos_dim=16, dropout=0.5, max_seq_len=512):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.pos_embed = nn.Embedding(max_seq_len, pos_dim)

        lstm_input_dim = word_dim + cnn_filters + pos_dim
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers,
                            bidirectional=True, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]

        w_emb = self.word_embed(word_ids)

        char_ids_flat = char_ids.view(-1, max_word_len)
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)
        c_features = c_cnn_out.view(batch_size, seq_len, -1)

        positions = torch.arange(seq_len, device=word_ids.device).unsqueeze(0).expand(batch_size, -1)
        pos_emb = self.pos_embed(positions)

        lstm_in = self.dropout(torch.cat([w_emb, c_features, pos_emb], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        emissions = self.hidden2tag(lstm_out)

        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')

        tags = self.crf.decode(emissions, mask=mask)
        return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS
# ==========================================
class HKAddressParserBiLSTM:
    def __init__(self, model_path):
        self.model_path = model_path
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        vocab_path = os.path.join(model_path, "vocabs.json")
        weights_path = os.path.join(model_path, "pytorch_model.bin")

        if not os.path.exists(vocab_path) or not os.path.exists(weights_path):
            raise FileNotFoundError(f"Model or vocabs not found in {model_path}.")

        with open(vocab_path, "r", encoding="utf-8") as f:
            vocabs = json.load(f)
        self.w2i = vocabs["w2i"]
        self.c2i = vocabs["c2i"]
        self.t2i = vocabs["t2i"]
        self.idx2tag = {int(v): k for k, v in self.t2i.items()}

        self.model = BiLSTM_CNN_CRF(
            vocab_size=len(self.w2i),
            char_vocab_size=len(self.c2i),
            num_tags=len(self.t2i),
            word_dim=WORD_EMBED_DIM,
            char_dim=CHAR_EMBED_DIM,
            cnn_filters=CHAR_CNN_FILTERS,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LSTM_LAYERS,
            pos_dim=POS_EMBED_DIM,
            dropout=DROPOUT
        )

        print(f"📦 Loading weights from {weights_path} ...")
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available():
            return torch.device("cpu")
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
                 "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id, max_free_mb = 0, 0
            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id, free_memory, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
                if gpu_util < 30 and free_memory > max_free_mb:
                    max_free_mb, best_id = free_memory, gpu_id
            return torch.device(f"cuda:{best_id}")
        except Exception:
            return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)
        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O":
                continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output, conf_output = {}, {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0
        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        """
        EXACT copy of the new training-script logic (NO reorder).
        """
        line1_keys = set()
        logic_keys_used = set()

        if "flat" in extracted_labels:
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels:
            line1_keys.add("floor")
            logic_keys_used.add("floor")

        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels

        # ---- new building / block / estate logic (from training V5) ----
        if has_block:
            # block always goes to micro
            line1_keys.add("block")
            logic_keys_used.add("block")
            if has_bldg and has_est:
                # block + building → micro, estate → macro
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
                # estate intentionally left out of line1_keys
            elif has_bldg and not has_est:
                # block → micro, building → macro
                # (do NOT add building_name)
                pass
            # else: only block → already handled
        elif has_bldg:
            # no block
            line1_keys.add("building_name")
            logic_keys_used.add("building_name")
            # if estate also exists it stays macro (not added)
        elif has_est:
            # no block, no building → estate becomes micro
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase:
                line1_keys.add("phase")
                logic_keys_used.add("phase")
        # ---------------------------------------------
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")

        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)

        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }

        token_groups = []
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys:
                token_groups.append("micro")
            elif mapped_tag not in ("o", "O"):
                token_groups.append("macro")
            else:
                token_groups.append("O")

        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro"
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)

        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None

        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]

            if group == "micro":
                if tag not in ("o", "O"):
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg:
                        curr_micro_seg.append(word)
                    else:
                        curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag not in ("o", "O"):
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg:
                        curr_macro_seg.append(word)
                    else:
                        curr_macro_seg, curr_macro_tag = [word], "o"

        if curr_micro_seg:
            micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg:
            macro_segments.append((curr_macro_tag, curr_macro_seg))

        # NO reorder (explicitly matching training)
        micro_string = "".join("".join(words) for _, words in micro_segments)
        macro_string = "".join("".join(words) for _, words in macro_segments)

        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()

        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)

        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)

        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ("district", "region", "sub_district"):
                    split_conf *= c

        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string

        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, full_addresses, batch_size=32):
        all_results = []
        for i in tqdm(range(0, len(full_addresses), batch_size), desc="Processing"):
            batch = full_addresses[i:i + batch_size]

            word_ids, char_ids, masks, batch_tokens = prepare_batch(batch, self.w2i, self.c2i)
            word_ids = word_ids.to(self.device)
            char_ids = char_ids.to(self.device)
            masks = masks.to(self.device)

            with torch.no_grad():
                prediction_ids_batch, emissions_batch = self.model(word_ids, char_ids, masks)
                batch_probs = torch.softmax(emissions_batch, dim=-1).cpu()

            for idx, address_str in enumerate(batch):
                prediction_ids = prediction_ids_batch[idx]
                item_probs = batch_probs[idx]
                tokens = batch_tokens[idx]

                # Character-level tag/conf mapping (same as training evaluate_on_test)
                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                start_indices = [m.start() for m in re.finditer(
                    r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', address_str)]

                for j, tag_id in enumerate(prediction_ids):
                    if j >= len(start_indices):
                        break
                    start_idx = start_indices[j]
                    tag = self.idx2tag[tag_id]
                    conf = item_probs[j, tag_id].item()
                    token_len = len(tokens[j])
                    for c in range(start_idx, min(start_idx + token_len, len(char_tags))):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                # Preserve trailing spaces
                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing = match.group(2)
                    start_idx = match.start(1)
                    tag = char_tags[start_idx] if start_idx < len(char_tags) else "O"
                    conf = char_confs[start_idx] if start_idx < len(char_confs) else 0.0
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(
                    extracted_raw, conf_raw)

                line1, line2, split_conf, used_keys = self._split_address(
                    extracted_mapped, address_str, parsed_entities, conf_mapped)

                all_results.append(
                    (address_str, extracted_mapped, conf_mapped, line1, line2, split_conf, used_keys)
                )
        return all_results

# ==========================================
# MAIN EXECUTION & EVALUATION
# ==========================================
def main():
    if not os.path.exists(MODEL_DIR) or not os.path.exists(TEST_FILE):
        print("❌ Error: Model directory or Test file not found.")
        return

    parser = HKAddressParserBiLSTM(model_path=MODEL_DIR)

    print(f"🚀 Loading dataset from {TEST_FILE}...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        test_data = [json.loads(line) for line in file if line.strip() and not line.startswith("#")]

    inputs = [item["input"].strip() for item in test_data]

    start_time = time.perf_counter()
    results = parser.parse_batch(inputs, batch_size=BATCH_SIZE)
    total_time = time.perf_counter() - start_time

    stats = {
        t: {
            "total_in_bin": 0,
            "logic_correct": 0,
            "line1_correct": 0,
            "line2_correct": 0,
            "full_correct": 0
        } for t in THRESHOLDS
    }
    excluded_count = 0

    print(f"✍️ Evaluating and writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (address, pred_tags, pred_confs, line1, line2, split_conf, used_keys) in enumerate(results):
            ground_truth_flat = flatten_json(test_data[idx].get("output", {}))

            # Corrupted GT check (same as training)
            norm_address = normalize_for_eval(address)
            is_corrupted = False
            for _, gt_val in ground_truth_flat.items():
                norm_gt_val = normalize_for_eval(gt_val)
                if norm_gt_val and norm_gt_val not in norm_address:
                    is_corrupted = True
                    break

            if is_corrupted:
                excluded_count += 1
                log.write(f"--- Result {idx + 1} [EXCLUDED: CORRUPTED DATA] ---\n")
                log.write(f"Original Input : {address}\n")
                log.write("Reason : Ground truth contains values not present in the input text.\n")
                log.write("-" * 50 + "\n")
                continue

            # Field correctness (aligned with training evaluate_on_test)
            pred_ff = normalize_for_eval(pred_tags.get("floor", "") + pred_tags.get("flat", ""))
            gt_ff = normalize_for_eval(ground_truth_flat.get("floor", "") + ground_truth_flat.get("flat", ""))

            p_bldg = normalize_for_eval(pred_tags.get("building_name", ""))
            p_est = normalize_for_eval(pred_tags.get("estate_name", ""))
            g_bldg = normalize_for_eval(ground_truth_flat.get("building_name", ""))
            g_est = normalize_for_eval(ground_truth_flat.get("estate_name", ""))

            field_correct = {}
            for field in ALL_FIELDS:
                p_norm = normalize_for_eval(pred_tags.get(field, ""))
                g_norm = normalize_for_eval(ground_truth_flat.get(field, ""))

                if p_norm == g_norm:
                    field_correct[field] = True
                elif field in ['floor', 'flat'] and pred_ff == gt_ff and pred_ff != "":
                    field_correct[field] = True
                elif field in ['building_name', 'estate_name'] and (p_bldg == g_est and p_est == g_bldg):
                    field_correct[field] = True
                else:
                    field_correct[field] = False

            is_logic_correct = all(field_correct.get(k, False) for k in used_keys) if used_keys else True

            line1_fields = ["flat", "floor", "block", "phase", "building_name", "estate_name"]
            is_l1_correct = all(field_correct[f] for f in line1_fields)

            line2_fields = ["village_name", "building_number", "street_name",
                            "sub_district", "district", "region"]
            is_l2_correct = all(field_correct[f] for f in line2_fields)

            is_full_correct = is_l1_correct and is_l2_correct

            for t in THRESHOLDS:
                if split_conf >= t:
                    stats[t]["total_in_bin"] += 1
                    if is_logic_correct:
                        stats[t]["logic_correct"] += 1
                    if is_l1_correct:
                        stats[t]["line1_correct"] += 1
                    if is_l2_correct:
                        stats[t]["line2_correct"] += 1
                    if is_full_correct:
                        stats[t]["full_correct"] += 1

            # Logging
            log.write(f"--- Result {idx + 1} ---\n")
            log.write(f"Original Input : {address}\n")
            log.write(f"Logic Keys Used : {used_keys}\n")
            log.write(f"Split Conf : {split_conf:.6f} (Product of keys used in split)\n")
            log.write(f"Output Line 1 : {line1}\n")
            log.write(f"Output Line 2 : {line2}\n")

            if is_full_correct:
                log.write("✅ EXACT MATCH\n")
            else:
                log.write("❌ MISMATCH FOUND\n")

            for field in ALL_FIELDS:
                pred_val = pred_tags.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                if pred_val or gt_val:
                    status = "✅" if field_correct[field] else "❌"
                    conf_str = f" conf={pred_confs.get(field, 0.0):.4f}"
                    log.write(f" {status} {field.upper()}:{conf_str}\n")
                    log.write(f" PRED: {pred_val if pred_val else '[None]'}\n")
                    log.write(f" TRUE: {gt_val if gt_val else '[None]'}\n")

            log.write("-" * 50 + "\n")

        # Summary table
        total_samples = len(inputs)
        valid_samples = total_samples - excluded_count

        def build_table_str():
            out = "\n" + "=" * 65 + "\n"
            out += "📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF V5)\n"
            out += "=" * 65 + "\n"
            out += f"Total Addresses Provided: {total_samples}\n"
            out += f"Excluded (Corrupted) : {excluded_count}\n"
            out += f"Total Valid Evaluated : {valid_samples}\n"
            out += f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n\n"

            for t in THRESHOLDS:
                title = "ALL PREDICTIONS (Threshold 0%)" if t == 0.0 else f"PREDICTIONS WITH ≥ {int(t*100)}% CONFIDENCE"
                out += "-" * 65 + "\n"
                out += f"🚀 {title}\n"
                out += "-" * 65 + "\n"

                total_in_bin = stats[t]["total_in_bin"]
                if total_in_bin > 0:
                    out += f"{'METRIC':<35} | {'ACCURACY (Correct / Total in Bin)'}\n"
                    out += "-" * 65 + "\n"

                    l_acc = (stats[t]["logic_correct"] / total_in_bin) * 100
                    l1_acc = (stats[t]["line1_correct"] / total_in_bin) * 100
                    l2_acc = (stats[t]["line2_correct"] / total_in_bin) * 100
                    full_acc = (stats[t]["full_correct"] / total_in_bin) * 100

                    out += f"{'Split Logic Determination Correct':<35} | {l_acc:>6.2f}% ({stats[t]['logic_correct']}/{total_in_bin})\n"
                    out += f"{'Line 1 (Micro) Components Correct':<35} | {l1_acc:>6.2f}% ({stats[t]['line1_correct']}/{total_in_bin})\n"
                    out += f"{'Line 2 (Macro) Components Correct':<35} | {l2_acc:>6.2f}% ({stats[t]['line2_correct']}/{total_in_bin})\n"
                    out += f"{'Full Address Perfect Match':<35} | {full_acc:>6.2f}% ({stats[t]['full_correct']}/{total_in_bin})\n\n"
                else:
                    out += f"No samples met the >= {int(t*100)}% confidence threshold.\n\n"
            return out

        table_output = build_table_str()
        print(table_output)
        log.write(table_output)

if __name__ == "__main__":
    main()

DEBUG: Using Device -> cuda:1
📦 Loading weights from ./bilstm_crf_modelV5/pytorch_model.bin ...
🚀 Loading dataset from data2/test.jsonl...


Processing: 100%|██████████| 1228/1228 [00:45<00:00, 27.28it/s]


✍️ Evaluating and writing results to address_split_results_bilstm_V5.log...

📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF V5)
Total Addresses Provided: 39277
Excluded (Corrupted) : 0
Total Valid Evaluated : 39277
⏱️ Total Inference runtime: 45.0156 seconds

-----------------------------------------------------------------
🚀 ALL PREDICTIONS (Threshold 0%)
-----------------------------------------------------------------
METRIC                              | ACCURACY (Correct / Total in Bin)
-----------------------------------------------------------------
Split Logic Determination Correct   |  98.80% (38806/39277)
Line 1 (Micro) Components Correct   |  99.00% (38885/39277)
Line 2 (Macro) Components Correct   |  99.32% (39011/39277)
Full Address Perfect Match          |  98.51% (38692/39277)

-----------------------------------------------------------------
🚀 PREDICTIONS WITH ≥ 20% CONFIDENCE
-----------------------------------------------------------------
METRIC                         

In [2]:
#!/usr/bin/env python3
"""
HK Address Parser - Final App with Integrated Evaluation (BiLSTM-CNN-CRF Version)
Evaluates address splitting logic into Line 1 / Line 2 strings.
This version is adapted for datasets where ground-truth "output" contains only
plain string "line1" and "line2" (no nested field tags).
It still runs the full BiLSTM-CNN-CRF tagger + split logic, then measures
string-level accuracy of the produced Line 1 / Line 2 against the GT strings,
binned by the split-aware confidence.
"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from torchcrf import CRF

# slower but works by disabling cuDNN completely
torch.backends.cudnn.enabled = False

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./bilstm_crf_modelV3_E10"
LOG_FILE = "address_split_results_bilstmV3_line_strings.log"
TEST_FILE = "data2/address_dataset.jsonl"          # change to your new dataset path if needed
BATCH_SIZE = 32
WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 512
NUM_LSTM_LAYERS = 2
THRESHOLDS = [0.0, 0.2, 0.3, 0.4, 0.50, 0.60, 0.70, 0.80]
ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ==========================================
# UTILITY FUNCTIONS FOR EVALUATION
# ==========================================
def normalize_for_eval(text):
    """Lowercases and removes ALL spaces/punctuation for strict string comparison."""
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

def tokenize_text(input_text):
    """Tokenize exactly as done in BiLSTM training."""
    tokens = []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        tokens.append(match.group())
    return tokens

def prepare_batch(batch_texts, w2i, c2i):
    """Pads and prepares tensors for the BiLSTM model."""
    batch_tokens = [tokenize_text(text) for text in batch_texts]
    max_seq_len = max(len(t) for t in batch_tokens)
    max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])

    b_words, b_chars, b_masks = [], [], []
    for tokens in batch_tokens:
        seq_len = len(tokens)
        word_ids = [w2i.get(t, w2i.get("<UNK>", 1)) for t in tokens]
        char_ids_list = [[c2i.get(c, c2i.get("<UNK>", 1)) for c in token] for token in tokens]

        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))

        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)

    return (
        torch.tensor(b_words, dtype=torch.long),
        torch.tensor(b_chars, dtype=torch.long),
        torch.tensor(b_masks, dtype=torch.bool),
        batch_tokens
    )

def extract_gt_lines(output_dict):
    """
    Extract ground-truth Line 1 and Line 2 strings.
    Supports the new format (plain strings) and gracefully falls back
    if the old nested-dict format is accidentally present.
    """
    if not isinstance(output_dict, dict):
        return "", ""

    l1 = output_dict.get("line1", "")
    l2 = output_dict.get("line2", "")

    # New format: already strings
    if isinstance(l1, str) and isinstance(l2, str):
        return l1.strip(), l2.strip()

    # Old nested format (dict of tags) – reconstruct a rough string for safety
    # (not expected for the new dataset, but prevents crashes)
    if isinstance(l1, dict) or isinstance(l2, dict):
        def dict_to_str(d):
            if not isinstance(d, dict):
                return str(d)
            # Prefer a canonical order if possible
            parts = []
            for k in ALL_FIELDS:
                if k in d and d[k]:
                    parts.append(str(d[k]))
            return "".join(parts) if parts else ""
        return dict_to_str(l1), dict_to_str(l2)

    return str(l1).strip(), str(l2).strip()

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)

        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers, bidirectional=True, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]

        w_emb = self.word_embed(word_ids)

        char_ids_flat = char_ids.view(-1, max_word_len)
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)

        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)
        c_features = c_cnn_out.view(batch_size, seq_len, -1)

        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)

        emissions = self.hidden2tag(lstm_out)

        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')

        tags = self.crf.decode(emissions, mask=mask)
        return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS (BiLSTM Version)
# ==========================================
class HKAddressParserBiLSTM:
    def __init__(self, model_path):
        self.model_path = model_path
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")
        vocab_path = os.path.join(model_path, "vocabs.json")
        weights_path = os.path.join(model_path, "pytorch_model.bin")

        if not os.path.exists(vocab_path) or not os.path.exists(weights_path):
            raise FileNotFoundError(f"Model or vocabs not found in {model_path}.")
        # Load Vocabs
        with open(vocab_path, "r", encoding="utf-8") as f:
            vocabs = json.load(f)
        self.w2i = vocabs["w2i"]
        self.c2i = vocabs["c2i"]
        self.t2i = vocabs["t2i"]
        self.idx2tag = {int(v): k for k, v in self.t2i.items()}

        # Initialize Model
        self.model = BiLSTM_CNN_CRF(
            vocab_size=len(self.w2i),
            char_vocab_size=len(self.c2i),
            num_tags=len(self.t2i),
            word_dim=WORD_EMBED_DIM,
            char_dim=CHAR_EMBED_DIM,
            cnn_filters=CHAR_CNN_FILTERS,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LSTM_LAYERS
        )

        print(f"📦 Loading weights from {weights_path} ...")
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available(): return torch.device("cpu")
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id, max_free_mb = 0, 0
            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id, free_memory, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
                if gpu_util < 30 and free_memory > max_free_mb:
                    max_free_mb, best_id = free_memory, gpu_id
            return torch.device(f"cuda:{best_id}")
        except Exception: return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)
        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])
        formatted_output, conf_output = {}, {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0
        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        """
        Splits address and calculates confidence strictly based on labels
        that influenced the split decision. Includes logic to correctly bind
        Building Numbers with Village and Street names, and separates Estate/Phase
        from Building/Block when both hierarchical levels exist.
        """
        line1_keys = set()
        logic_keys_used = set()

        # 1. Micro elements
        if "flat" in extracted_labels:
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels:
            line1_keys.add("floor")
            logic_keys_used.add("floor")
        # 2. Structural elements (Hierarchical separation logic)
        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels

        # RULE 1: If Building OR Block exists, they take priority for Line 1 (micro).
        if has_bldg or has_block:
            if has_bldg:
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
            if has_block:
                line1_keys.add("block")
                logic_keys_used.add("block")

        # RULE 2: If NO Building or Block, but Estate exists, Estate acts as the building (Line 1).
        elif has_est:
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase:
                line1_keys.add("phase")
                logic_keys_used.add("phase")

        # RULE 3: Fallbacks for Village / Street / Building Number
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")
        # 3. Assign tokens
        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        token_groups = []
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }

        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys: token_groups.append("micro")
            elif mapped_tag != "o" and mapped_tag != "O": token_groups.append("macro")
            else: token_groups.append("O")

        # Resolve punctuation
        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro"
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)

        # Group adjacent tokens into segments by their tag
        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None

        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]

            if group == "micro":
                if tag != 'o' and tag != 'O':
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg: curr_micro_seg.append(word)
                    else: curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag != 'o' and tag != 'O':
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg: curr_macro_seg.append(word)
                    else: curr_macro_seg, curr_macro_tag = [word], "o"

        if curr_micro_seg: micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg: macro_segments.append((curr_macro_tag, curr_macro_seg))

        # ---------------------------------------------------------
        # REMOVED reorder_segments() - Strings now stay exactly
        # in the order the user typed them!
        # def reorder_segments(segments, is_chinese):
        #     """Moves the building_number chunk to sit directly next to the primary grouping element"""
        #     bldg_no_idx = next((i for i, s in enumerate(segments) if s[0] == "building_number"), -1)
        #     if bldg_no_idx == -1: return segments
            
        #     bldg_no_seg = segments.pop(bldg_no_idx)
        #     target_tags = ["village_name", "street_name", "estate_name", "building_name"]
            
        #     if is_chinese:
        #         target_idx = -1
        #         for i, s in enumerate(segments):
        #             if s[0] in target_tags: target_idx = i
        #         if target_idx != -1: segments.insert(target_idx + 1, bldg_no_seg)
        #         else: segments.insert(0, bldg_no_seg)
        #     else:
        #         target_idx = -1
        #         for i, s in enumerate(segments):
        #             if s[0] in target_tags:
        #                 target_idx = i
        #                 break
        #         if target_idx != -1: segments.insert(target_idx, bldg_no_seg)
        #         else: segments.append(bldg_no_seg)
        #     return segments

        # micro_segments = reorder_segments(micro_segments, is_chinese)
        # macro_segments = reorder_segments(macro_segments, is_chinese)
        # ---------------------------------------------------------
        
        micro_string = "".join("".join(words) for tag, words in micro_segments)
        macro_string = "".join("".join(words) for tag, words in macro_segments)

        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()
        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)

        # Calculate Logic Confidence
        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)

        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ["district", "region", "sub_district"]: split_conf *= c
        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string
        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, full_addresses, batch_size=32):
        all_results = []
        for i in tqdm(range(0, len(full_addresses), batch_size), desc="Processing"):
            batch = full_addresses[i : i + batch_size]

            word_ids, char_ids, masks, batch_tokens = prepare_batch(batch, self.w2i, self.c2i)
            word_ids = word_ids.to(self.device)
            char_ids = char_ids.to(self.device)
            masks = masks.to(self.device)
            with torch.no_grad():
                prediction_ids_batch, emissions_batch = self.model(word_ids, char_ids, masks)
                batch_probs = torch.softmax(emissions_batch, dim=-1).cpu()
            for idx, address_str in enumerate(batch):
                prediction_ids = prediction_ids_batch[idx]
                item_probs = batch_probs[idx]
                tokens = batch_tokens[idx]

                # 1. Map tokens back to the original string character by character
                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)

                start_indices = [match.start() for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', address_str)]

                for j, tag_id in enumerate(prediction_ids):
                    if j >= len(start_indices): break
                    start_idx = start_indices[j]
                    token_str = tokens[j]

                    tag = self.idx2tag[tag_id]
                    conf = item_probs[j, tag_id].item()
                    for c in range(start_idx, start_idx + len(token_str)):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf
                # 2. Extract with trailing spaces to preserve formatting
                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)
                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })
                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)

                # Split and get logic-aware confidence
                line1, line2, split_conf, used_keys = self._split_address(extracted_mapped, address_str, parsed_entities, conf_mapped)
                all_results.append((address_str, extracted_mapped, conf_mapped, line1, line2, split_conf, used_keys))
        return all_results

# ==========================================
# MAIN EXECUTION & EVALUATION LOOP
# ==========================================
def main():
    if not os.path.exists(MODEL_DIR) or not os.path.exists(TEST_FILE):
        print("❌ Error: Model directory or Test file not found.")
        print(f"   MODEL_DIR = {MODEL_DIR}")
        print(f"   TEST_FILE = {TEST_FILE}")
        return
    parser = HKAddressParserBiLSTM(model_path=MODEL_DIR)
    print(f"🚀 Loading dataset from {TEST_FILE}...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        test_data = [json.loads(line) for line in file if line.strip() and not line.startswith("#")]

    inputs = [item["input"].strip() for item in test_data]

    start_time = time.perf_counter()
    results = parser.parse_batch(inputs, batch_size=BATCH_SIZE)
    total_time = time.perf_counter() - start_time

    # Initialize Metrics Structure (string-level only)
    stats = {
        t: {
            "total_in_bin": 0,
            "line1_correct": 0,
            "line2_correct": 0,
            "full_correct": 0
        } for t in THRESHOLDS
    }
    excluded_count = 0
    print(f"✍️ Evaluating and writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (address, pred_tags, pred_confs, line1, line2, split_conf, used_keys) in enumerate(results):
            gt_output = test_data[idx].get("output", {})
            gt_line1, gt_line2 = extract_gt_lines(gt_output)

            # --- CHECK FOR CORRUPTED GROUND TRUTH ---
            # A sample is considered corrupted if the content of the GT lines
            # cannot be found inside the original input (after normalisation).
            norm_address = normalize_for_eval(address)
            norm_gt1 = normalize_for_eval(gt_line1)
            norm_gt2 = normalize_for_eval(gt_line2)
            is_corrupted = False
            if (norm_gt1 and norm_gt1 not in norm_address) or (norm_gt2 and norm_gt2 not in norm_address):
                is_corrupted = True

            if is_corrupted:
                excluded_count += 1
                log.write(f"--- Result {idx + 1} [EXCLUDED: CORRUPTED DATA] ---\n")
                log.write(f"Original Input  : {address}\n")
                log.write(f"GT Line 1       : {gt_line1}\n")
                log.write(f"GT Line 2       : {gt_line2}\n")
                log.write("Reason          : Ground truth line content not present in the input text.\n")
                log.write("-" * 50 + "\n")
                continue

            # --- STRING-LEVEL CORRECTNESS ---
            is_l1_correct = normalize_for_eval(line1) == normalize_for_eval(gt_line1)
            is_l2_correct = normalize_for_eval(line2) == normalize_for_eval(gt_line2)
            is_full_correct = is_l1_correct and is_l2_correct

            # --- BIN THE ACCURACY BY CONFIDENCE ---
            for t in THRESHOLDS:
                if split_conf >= t:
                    stats[t]["total_in_bin"] += 1
                    if is_l1_correct: stats[t]["line1_correct"] += 1
                    if is_l2_correct: stats[t]["line2_correct"] += 1
                    if is_full_correct: stats[t]["full_correct"] += 1

            # --- WRITE FULL LOGS ---
            log.write(f"--- Result {idx + 1} ---\n")
            log.write(f"Original Input  : {address}\n")
            log.write(f"Logic Keys Used : {used_keys}\n")
            log.write(f"Split Conf      : {split_conf:.6f} (Product of keys used in split)\n")
            log.write(f"PRED Line 1     : {line1}\n")
            log.write(f"GT   Line 1     : {gt_line1}\n")
            log.write(f"PRED Line 2     : {line2}\n")
            log.write(f"GT   Line 2     : {gt_line2}\n")

            if is_full_correct:
                log.write("✅ EXACT MATCH (both lines)\n")
            else:
                log.write("❌ MISMATCH FOUND\n")
                if not is_l1_correct:
                    log.write("   → Line 1 mismatch\n")
                if not is_l2_correct:
                    log.write("   → Line 2 mismatch\n")

            # Still dump the predicted field tags + confidence for debugging
            log.write("Predicted components:\n")
            for field in ALL_FIELDS:
                pred_val = pred_tags.get(field, "")
                if pred_val:
                    conf_str = f"  conf={pred_confs.get(field, 0.0):.4f}"
                    log.write(f"  {field.upper()}:{conf_str}\n")
                    log.write(f"     PRED: {pred_val}\n")

            log.write("-" * 50 + "\n")

        # --- WRITE SUMMARY TABLE ---
        total_samples = len(inputs)
        valid_samples = total_samples - excluded_count

        def build_table_str():
            out = "\n" + "=" * 65 + "\n"
            out += "📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF – LINE STRING EVAL)\n"
            out += "=" * 65 + "\n"
            out += f"Total Addresses Provided: {total_samples}\n"
            out += f"Excluded (Corrupted)    : {excluded_count}\n"
            out += f"Total Valid Evaluated   : {valid_samples}\n"
            out += f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n\n"
            for t in THRESHOLDS:
                title = "ALL PREDICTIONS (Threshold 0%)" if t == 0.0 else f"PREDICTIONS WITH ≥ {int(t*100)}% CONFIDENCE"
                out += "-" * 65 + "\n"
                out += f"🚀 {title}\n"
                out += "-" * 65 + "\n"

                total_in_bin = stats[t]["total_in_bin"]
                if total_in_bin > 0:
                    out += f"{'METRIC':<40} | {'ACCURACY (Correct / Total in Bin)'}\n"
                    out += "-" * 65 + "\n"

                    l1_acc = (stats[t]["line1_correct"] / total_in_bin) * 100
                    l2_acc = (stats[t]["line2_correct"] / total_in_bin) * 100
                    full_acc = (stats[t]["full_correct"] / total_in_bin) * 100

                    out += f"{'Line 1 (string) Correct':<40} | {l1_acc:>6.2f}%  ({stats[t]['line1_correct']}/{total_in_bin})\n"
                    out += f"{'Line 2 (string) Correct':<40} | {l2_acc:>6.2f}%  ({stats[t]['line2_correct']}/{total_in_bin})\n"
                    out += f"{'Full Address Perfect Match (both lines)':<40} | {full_acc:>6.2f}%  ({stats[t]['full_correct']}/{total_in_bin})\n\n"
                else:
                    out += f"No samples met the >= {int(t*100)}% confidence threshold.\n\n"
            return out
        table_output = build_table_str()
        print(table_output)
        log.write(table_output)

if __name__ == "__main__":
    main()

DEBUG: Using Device -> cuda:0
📦 Loading weights from ./bilstm_crf_modelV3_E10/pytorch_model.bin ...
🚀 Loading dataset from data2/address_dataset.jsonl...


Processing:   0%|          | 0/57 [00:00<?, ?it/s]

✍️ Evaluating and writing results to address_split_results_bilstmV3_line_strings.log...

📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF – LINE STRING EVAL)
Total Addresses Provided: 1813
Excluded (Corrupted)    : 198
Total Valid Evaluated   : 1615
⏱️ Total Inference runtime: 2.1745 seconds

-----------------------------------------------------------------
🚀 ALL PREDICTIONS (Threshold 0%)
-----------------------------------------------------------------
METRIC                                   | ACCURACY (Correct / Total in Bin)
-----------------------------------------------------------------
Line 1 (string) Correct                  |  51.89%  (838/1615)
Line 2 (string) Correct                  |  52.01%  (840/1615)
Full Address Perfect Match (both lines)  |  51.89%  (838/1615)

-----------------------------------------------------------------
🚀 PREDICTIONS WITH ≥ 20% CONFIDENCE
-----------------------------------------------------------------
METRIC                                   | AC

In [6]:
#!/usr/bin/env python3
"""
HK Address Parser - Final App with Integrated Evaluation (BiLSTM-CNN-CRF Version)
Evaluates address splitting into Line 1 / Line 2 strings.

Supports test file in TXT format (one sample per line):
    input_address | GT_line1 | GT_line2

It still runs the full BiLSTM-CNN-CRF tagger + split logic, then measures
string-level accuracy of the produced Line 1 / Line 2 against the GT strings,
binned by the split-aware confidence.
"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from torchcrf import CRF

# ==========================================
# CONFIGURATIONS
# ==========================================
MODEL_DIR = "./bilstm_crf_modelV3"
LOG_FILE = "address_split_results_bilstmV3_line_strings.log"
TEST_FILE = "data2/output__format_sampled_2.txt"          # <-- change to your .txt path
BATCH_SIZE = 32
WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 512
NUM_LSTM_LAYERS = 2
THRESHOLDS = [0.0, 0.2, 0.3, 0.4, 0.50, 0.60, 0.70, 0.80]
ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ==========================================
# UTILITY FUNCTIONS FOR EVALUATION
# ==========================================
def normalize_for_eval(text):
    """Lowercases and removes ALL spaces/punctuation for strict string comparison."""
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

def content_covered(gt_text, address_norm):
    """
    Corruption check that allows reordering / reformatting.
    Requires that every character appearing in the GT also appears
    in the input (with at least the same frequency). Order does NOT matter.
    Empty GT is always considered covered.
    """
    from collections import Counter
    gt_norm = normalize_for_eval(gt_text)
    if not gt_norm:
        return True
    gt_cnt = Counter(gt_norm)
    addr_cnt = Counter(address_norm)
    return all(addr_cnt[c] >= cnt for c, cnt in gt_cnt.items())

def tokenize_text(input_text):
    """Tokenize exactly as done in BiLSTM training."""
    tokens = []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        tokens.append(match.group())
    return tokens

def prepare_batch(batch_texts, w2i, c2i):
    """Pads and prepares tensors for the BiLSTM model."""
    batch_tokens = [tokenize_text(text) for text in batch_texts]
    max_seq_len = max(len(t) for t in batch_tokens) if batch_tokens else 1
    max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])

    b_words, b_chars, b_masks = [], [], []
    for tokens in batch_tokens:
        seq_len = len(tokens)
        word_ids = [w2i.get(t, w2i.get("<UNK>", 1)) for t in tokens]
        char_ids_list = [[c2i.get(c, c2i.get("<UNK>", 1)) for c in token] for token in tokens]

        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))

        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)

    return (
        torch.tensor(b_words, dtype=torch.long),
        torch.tensor(b_chars, dtype=torch.long),
        torch.tensor(b_masks, dtype=torch.bool),
        batch_tokens
    )

def load_txt_test_file(path):
    """
    Load test file in the format:
        input_address | GT_line1 | GT_line2
    Returns list of dicts compatible with the rest of the pipeline:
        {"input": ..., "output": {"line1": ..., "line2": ...}}
    """
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, raw in enumerate(f, 1):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            # Split on " | " (space-pipe-space) first, fallback to plain "|"
            if " | " in line:
                parts = [p.strip() for p in line.split(" | ")]
            else:
                parts = [p.strip() for p in line.split("|")]

            if len(parts) < 2:
                print(f"⚠️  Skipping malformed line {line_no}: {line[:80]}...")
                continue

            inp = parts[0]
            gt_l1 = parts[1] if len(parts) > 1 else ""
            gt_l2 = parts[2] if len(parts) > 2 else ""

            data.append({
                "input": inp,
                "output": {
                    "line1": gt_l1,
                    "line2": gt_l2
                }
            })
    return data

def extract_gt_lines(output_dict):
    """Extract ground-truth Line 1 and Line 2 strings."""
    if not isinstance(output_dict, dict):
        return "", ""
    l1 = output_dict.get("line1", "")
    l2 = output_dict.get("line2", "")
    return str(l1).strip(), str(l2).strip()

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)

        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers, bidirectional=True, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]

        w_emb = self.word_embed(word_ids)

        char_ids_flat = char_ids.view(-1, max_word_len)
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)

        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)
        c_features = c_cnn_out.view(batch_size, seq_len, -1)

        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)

        emissions = self.hidden2tag(lstm_out)

        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')

        tags = self.crf.decode(emissions, mask=mask)
        return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS (BiLSTM Version)
# ==========================================
class HKAddressParserBiLSTM:
    def __init__(self, model_path):
        self.model_path = model_path
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")
        vocab_path = os.path.join(model_path, "vocabs.json")
        weights_path = os.path.join(model_path, "pytorch_model.bin")

        if not os.path.exists(vocab_path) or not os.path.exists(weights_path):
            raise FileNotFoundError(f"Model or vocabs not found in {model_path}.")
        with open(vocab_path, "r", encoding="utf-8") as f:
            vocabs = json.load(f)
        self.w2i = vocabs["w2i"]
        self.c2i = vocabs["c2i"]
        self.t2i = vocabs["t2i"]
        self.idx2tag = {int(v): k for k, v in self.t2i.items()}

        self.model = BiLSTM_CNN_CRF(
            vocab_size=len(self.w2i),
            char_vocab_size=len(self.c2i),
            num_tags=len(self.t2i),
            word_dim=WORD_EMBED_DIM,
            char_dim=CHAR_EMBED_DIM,
            cnn_filters=CHAR_CNN_FILTERS,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LSTM_LAYERS
        )

        print(f"📦 Loading weights from {weights_path} ...")
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available(): return torch.device("cpu")
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id, max_free_mb = 0, 0
            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id, free_memory, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
                if gpu_util < 30 and free_memory > max_free_mb:
                    max_free_mb, best_id = free_memory, gpu_id
            return torch.device(f"cuda:{best_id}")
        except Exception: return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)
        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])
        formatted_output, conf_output = {}, {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0
        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        line1_keys = set()
        logic_keys_used = set()

        if "flat" in extracted_labels:
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels:
            line1_keys.add("floor")
            logic_keys_used.add("floor")

        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels

        if has_bldg or has_block:
            if has_bldg:
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
            if has_block:
                line1_keys.add("block")
                logic_keys_used.add("block")
        elif has_est:
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase:
                line1_keys.add("phase")
                logic_keys_used.add("phase")
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")

        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        token_groups = []
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }

        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys: token_groups.append("micro")
            elif mapped_tag != "o" and mapped_tag != "O": token_groups.append("macro")
            else: token_groups.append("O")

        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro"
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)

        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None

        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]

            if group == "micro":
                if tag != 'o' and tag != 'O':
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg: curr_micro_seg.append(word)
                    else: curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag != 'o' and tag != 'O':
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg: curr_macro_seg.append(word)
                    else: curr_macro_seg, curr_macro_tag = [word], "o"

        if curr_micro_seg: micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg: macro_segments.append((curr_macro_tag, curr_macro_seg))

        # ---------------------------------------------------------
        # REMOVED reorder_segments() - Strings now stay exactly
        # in the order the user typed them!
        # def reorder_segments(segments, is_chinese):
        #     """Moves the building_number chunk to sit directly next to the primary grouping element"""
        #     bldg_no_idx = next((i for i, s in enumerate(segments) if s[0] == "building_number"), -1)
        #     if bldg_no_idx == -1: return segments
            
        #     bldg_no_seg = segments.pop(bldg_no_idx)
        #     target_tags = ["village_name", "street_name", "estate_name", "building_name"]
            
        #     if is_chinese:
        #         target_idx = -1
        #         for i, s in enumerate(segments):
        #             if s[0] in target_tags: target_idx = i
        #         if target_idx != -1: segments.insert(target_idx + 1, bldg_no_seg)
        #         else: segments.insert(0, bldg_no_seg)
        #     else:
        #         target_idx = -1
        #         for i, s in enumerate(segments):
        #             if s[0] in target_tags:
        #                 target_idx = i
        #                 break
        #         if target_idx != -1: segments.insert(target_idx, bldg_no_seg)
        #         else: segments.append(bldg_no_seg)
        #     return segments

        # micro_segments = reorder_segments(micro_segments, is_chinese)
        # macro_segments = reorder_segments(macro_segments, is_chinese)
        # ---------------------------------------------------------
        micro_string = "".join("".join(words) for tag, words in micro_segments)
        macro_string = "".join("".join(words) for tag, words in macro_segments)

        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()
        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)

        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)

        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ["district", "region", "sub_district"]: split_conf *= c
        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string
        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, full_addresses, batch_size=32):
        all_results = []
        for i in tqdm(range(0, len(full_addresses), batch_size), desc="Processing"):
            batch = full_addresses[i : i + batch_size]

            word_ids, char_ids, masks, batch_tokens = prepare_batch(batch, self.w2i, self.c2i)
            word_ids = word_ids.to(self.device)
            char_ids = char_ids.to(self.device)
            masks = masks.to(self.device)
            with torch.no_grad():
                prediction_ids_batch, emissions_batch = self.model(word_ids, char_ids, masks)
                batch_probs = torch.softmax(emissions_batch, dim=-1).cpu()
            for idx, address_str in enumerate(batch):
                prediction_ids = prediction_ids_batch[idx]
                item_probs = batch_probs[idx]
                tokens = batch_tokens[idx]

                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)

                start_indices = [match.start() for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', address_str)]

                for j, tag_id in enumerate(prediction_ids):
                    if j >= len(start_indices): break
                    start_idx = start_indices[j]
                    token_str = tokens[j]

                    tag = self.idx2tag[tag_id]
                    conf = item_probs[j, tag_id].item()
                    for c in range(start_idx, start_idx + len(token_str)):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf
                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)
                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })
                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)

                line1, line2, split_conf, used_keys = self._split_address(extracted_mapped, address_str, parsed_entities, conf_mapped)
                all_results.append((address_str, extracted_mapped, conf_mapped, line1, line2, split_conf, used_keys))
        return all_results

# ==========================================
# MAIN EXECUTION & EVALUATION LOOP
# ==========================================
def main():
    if not os.path.exists(MODEL_DIR) or not os.path.exists(TEST_FILE):
        print("❌ Error: Model directory or Test file not found.")
        print(f"   MODEL_DIR = {MODEL_DIR}")
        print(f"   TEST_FILE = {TEST_FILE}")
        return

    parser = HKAddressParserBiLSTM(model_path=MODEL_DIR)
    print(f"🚀 Loading dataset from {TEST_FILE}...")
    test_data = load_txt_test_file(TEST_FILE)
    print(f"   Loaded {len(test_data)} samples.")

    inputs = [item["input"].strip() for item in test_data]

    start_time = time.perf_counter()
    results = parser.parse_batch(inputs, batch_size=BATCH_SIZE)
    total_time = time.perf_counter() - start_time

    stats = {
        t: {
            "total_in_bin": 0,
            "line1_correct": 0,
            "line2_correct": 0,
            "full_correct": 0
        } for t in THRESHOLDS
    }
    excluded_count = 0
    print(f"✍️ Evaluating and writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (address, pred_tags, pred_confs, line1, line2, split_conf, used_keys) in enumerate(results):
            gt_output = test_data[idx].get("output", {})
            gt_line1, gt_line2 = extract_gt_lines(gt_output)

            # Soft corruption check (order / contiguity independent)
            # GT lines in the TXT file are often reformatted/reordered, so the
            # old "entire normalised line must be a contiguous substring" test
            # incorrectly marks many valid samples as corrupted.
            norm_address = normalize_for_eval(address)
            is_corrupted = (
                not content_covered(gt_line1, norm_address) or
                not content_covered(gt_line2, norm_address)
            )

            if is_corrupted:
                excluded_count += 1
                log.write(f"--- Result {idx + 1} [EXCLUDED: CORRUPTED DATA] ---\n")
                log.write(f"Original Input  : {address}\n")
                log.write(f"GT Line 1       : {gt_line1}\n")
                log.write(f"GT Line 2       : {gt_line2}\n")
                log.write("Reason          : Some characters in GT lines do not appear in the input text.\n")
                log.write("-" * 50 + "\n")
                continue

            # String-level correctness
            is_l1_correct = normalize_for_eval(line1) == normalize_for_eval(gt_line1)
            is_l2_correct = normalize_for_eval(line2) == normalize_for_eval(gt_line2)
            is_full_correct = is_l1_correct and is_l2_correct

            for t in THRESHOLDS:
                if split_conf >= t:
                    stats[t]["total_in_bin"] += 1
                    if is_l1_correct: stats[t]["line1_correct"] += 1
                    if is_l2_correct: stats[t]["line2_correct"] += 1
                    if is_full_correct: stats[t]["full_correct"] += 1

            # Logging
            log.write(f"--- Result {idx + 1} ---\n")
            log.write(f"Original Input  : {address}\n")
            log.write(f"Logic Keys Used : {used_keys}\n")
            log.write(f"Split Conf      : {split_conf:.6f} (Product of keys used in split)\n")
            log.write(f"PRED Line 1     : {line1}\n")
            log.write(f"GT   Line 1     : {gt_line1}\n")
            log.write(f"PRED Line 2     : {line2}\n")
            log.write(f"GT   Line 2     : {gt_line2}\n")

            if is_full_correct:
                log.write("✅ EXACT MATCH (both lines)\n")
            else:
                log.write("❌ MISMATCH FOUND\n")
                if not is_l1_correct:
                    log.write("   → Line 1 mismatch\n")
                if not is_l2_correct:
                    log.write("   → Line 2 mismatch\n")

            log.write("Predicted components:\n")
            for field in ALL_FIELDS:
                pred_val = pred_tags.get(field, "")
                if pred_val:
                    conf_str = f"  conf={pred_confs.get(field, 0.0):.4f}"
                    log.write(f"  {field.upper()}:{conf_str}\n")
                    log.write(f"     PRED: {pred_val}\n")

            log.write("-" * 50 + "\n")

        total_samples = len(inputs)
        valid_samples = total_samples - excluded_count

        def build_table_str():
            out = "\n" + "=" * 65 + "\n"
            out += "📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF – LINE STRING EVAL)\n"
            out += "=" * 65 + "\n"
            out += f"Total Addresses Provided: {total_samples}\n"
            out += f"Excluded (Corrupted)    : {excluded_count}\n"
            out += f"Total Valid Evaluated   : {valid_samples}\n"
            out += f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n\n"
            for t in THRESHOLDS:
                title = "ALL PREDICTIONS (Threshold 0%)" if t == 0.0 else f"PREDICTIONS WITH ≥ {int(t*100)}% CONFIDENCE"
                out += "-" * 65 + "\n"
                out += f"🚀 {title}\n"
                out += "-" * 65 + "\n"

                total_in_bin = stats[t]["total_in_bin"]
                if total_in_bin > 0:
                    out += f"{'METRIC':<40} | {'ACCURACY (Correct / Total in Bin)'}\n"
                    out += "-" * 65 + "\n"

                    l1_acc = (stats[t]["line1_correct"] / total_in_bin) * 100
                    l2_acc = (stats[t]["line2_correct"] / total_in_bin) * 100
                    full_acc = (stats[t]["full_correct"] / total_in_bin) * 100

                    out += f"{'Line 1 (string) Correct':<40} | {l1_acc:>6.2f}%  ({stats[t]['line1_correct']}/{total_in_bin})\n"
                    out += f"{'Line 2 (string) Correct':<40} | {l2_acc:>6.2f}%  ({stats[t]['line2_correct']}/{total_in_bin})\n"
                    out += f"{'Full Address Perfect Match (both lines)':<40} | {full_acc:>6.2f}%  ({stats[t]['full_correct']}/{total_in_bin})\n\n"
                else:
                    out += f"No samples met the >= {int(t*100)}% confidence threshold.\n\n"
            return out
        table_output = build_table_str()
        print(table_output)
        log.write(table_output)

if __name__ == "__main__":
    main()

DEBUG: Using Device -> cuda:0
📦 Loading weights from ./bilstm_crf_modelV3/pytorch_model.bin ...
🚀 Loading dataset from data2/output__format_sampled_2.txt...
   Loaded 498 samples.


Processing:   0%|          | 0/16 [00:00<?, ?it/s]

✍️ Evaluating and writing results to address_split_results_bilstmV3_line_strings.log...

📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF – LINE STRING EVAL)
Total Addresses Provided: 498
Excluded (Corrupted)    : 23
Total Valid Evaluated   : 475
⏱️ Total Inference runtime: 0.6490 seconds

-----------------------------------------------------------------
🚀 ALL PREDICTIONS (Threshold 0%)
-----------------------------------------------------------------
METRIC                                   | ACCURACY (Correct / Total in Bin)
-----------------------------------------------------------------
Line 1 (string) Correct                  |  56.63%  (269/475)
Line 2 (string) Correct                  |  52.84%  (251/475)
Full Address Perfect Match (both lines)  |  48.63%  (231/475)

-----------------------------------------------------------------
🚀 PREDICTIONS WITH ≥ 20% CONFIDENCE
-----------------------------------------------------------------
METRIC                                   | ACCURACY